## 📝 Task info:

### AI Hospital Readmission & Length-of-Stay
## Intelligence Platform

## Problem Name: AI 30-Day Hospital Readmission Risk Prediction
- ML Task: Binary Classification
- Problem Statement: Predict whether an eligible hospital encounter will be followed by a 30-day readmission, so the care-management team can         - prioritize patients for additional follow-up.
## Target (y):
Existing column: readmitted
## Input Features (X): 
age, gender, race, admission_type_id, admission_source_id, time_in_hospital, medical_specialty, number_outpatient, number_emergency, number_inpatient, number_diagnoses, diag_1, diag_2, diag_3, max_glu_serum, A1Cresult, num_medications, num_lab_procedures, num_procedures, insulin, diabetesMed, change
## Feature Engineering:
prior_utilization, high_inpatient_utilization, high_emergency_utilization, diagnosis_complexity, medication_burden_group, age_group, glucose_abnormal_flag, insulin_change_flag, diag_1_category, diag_2_category, diag_3_category, multiple_prior_encounters_flag, high_diagnosis_burden_flag

## Why This Problem:
This is by far the best-supported problem in your dataset.
Your EDA showed previous inpatient visits increasing from 8.44% observed 30-day readmission at 0 visits to 31.40% at 5 visits and 44.37% at 8 visits. Previous emergency visits also increased from 10.47% at 0 visits to 27.08% at 5+ visits.


Suggested Algorithms:
•	Logistic Regression 
•	Random Forest Classifier 
•	HistGradientBoostingClassifier 
•	Gradient Boosting Classifier 
Evaluation Metrics:
•	PR-AUC 
•	Recall 
•	Precision 
•	F1-score 
•	ROC-AUC 
•	Confusion Matrix 
•	Calibration / Brier score


## 💻 Code Work — Task 2: 30-Day Readmission Classification

In [1]:
## Import Required Libraries
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats

## Install MySQL Connector

In [2]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
## Import and Connect to MySQL
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="hospital_readmission"
)

connection.is_connected()

True

In [4]:
## Then Load the validated table
df_model_classification = pd.read_sql(
    "SELECT * FROM diabetic_data",
    connection
)

df_model_classification.shape

C:\Users\chinn\AppData\Local\Temp\ipykernel_16256\817059187.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_model_classification = pd.read_sql(


(101766, 50)

In [5]:
ids_mapping_classification = pd.read_sql(
    "SELECT * FROM ids_mapping_raw",
    connection
)

ids_mapping_classification.shape

C:\Users\chinn\AppData\Local\Temp\ipykernel_16256\271552360.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ids_mapping_classification = pd.read_sql(


(67, 2)

In [6]:
## Create the Three Mapping Sections
admission_type_mapping = ids_mapping_classification.iloc[:8].copy()

discharge_disposition_mapping = ids_mapping_classification.iloc[9:39].copy()

admission_source_mapping = ids_mapping_classification.iloc[41:67].copy()

In [7]:
## Verify the mappings
mapping_summary = pd.DataFrame({
    "Mapping": [
        "Admission Type",
        "Discharge Disposition",
        "Admission Source"
    ],
    "Rows": [
        len(admission_type_mapping),
        len(discharge_disposition_mapping),
        len(admission_source_mapping)
    ]
})

mapping_summary

,Mapping,Rows
0,Admission Type,8
1,Discharge Disposition,30
2,Admission Source,26


## 👩🏻‍💻 1. Modeling
## 🛠️ 1.1 Pre-Requisites
## 🎯 1.1.1 X & y:
- Picking y output col for Predictive Modelling & considering remaining cols are X

- For this project, we should not simply take the existing readmitted column as-is, because our business problem is specifically 30-day readmission. Therefore, readmitted must first be converted into the binary target target_readmit_30d, while terminal discharge encounters are excluded from the eligible population. The approved Classification input columns are then used as X, with discharge_disposition_id excluded because it is used for eligibility and can introduce leakage.

## 1.1.1 X & y — Define Target and Features

# Target (y):

- <30 → 1 = readmitted within 30 days
- <30 → 0
- NO → 0

## Create the eligible modeling dataset
- We need to establish the correct population before creating X and y. Otherwise, the target distribution and subsequent train/test split could include encounters that should never have entered the readmission prediction problem.

In [8]:
# Terminal discharge dispositions that are not eligible for 30-day readmission modeling
terminal_disposition_ids = [11, 13, 14, 19, 20, 21]

df_readmission_eligible = df_model_classification[
    ~df_model_classification["discharge_disposition_id"].isin(
        terminal_disposition_ids
    )
].copy()

df_readmission_eligible.shape

(99343, 50)

## Create binary target
- Now we convert the original three-class readmitted outcome into the business target required by our Classification problem.

In [9]:
df_readmission_eligible["target_readmit_30d"] = (
    df_readmission_eligible["readmitted"] == "<30"
).astype(int)

df_readmission_eligible["target_readmit_30d"].value_counts()

target_readmit_30d
0    88029
1    11314
Name: count, dtype: int64

## Define X and y

In [10]:
classification_features = [
    "age",
    "gender",
    "race",
    "admission_type_id",
    "admission_source_id",
    "time_in_hospital",
    "medical_specialty",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
    "num_medications",
    "num_lab_procedures",
    "num_procedures",
    "insulin",
    "diabetesMed",
    "change"
]

X_classification = df_readmission_eligible[classification_features].copy()

y_classification = df_readmission_eligible["target_readmit_30d"].copy()

X_classification.shape, y_classification.shape

((99343, 22), (99343,))

## 1.1.2 Train-Test Split


- Because the target is imbalanced (11.39% positive), we should use a stratified split. This keeps approximately the same proportion of 30-day readmissions in both training and testing, while keeping the test set completely unseen during model development.

Use 80% training / 20% testing, matching our Regression workflow:

In [11]:
# 1.1.2 Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_classification,
    y_classification,
    test_size=0.20,
    random_state=42,
    stratify=y_classification
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((79474, 22), (19869, 22), (79474,), (19869,))

## Verify target distribution after stratification


In [12]:
split_target_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Total_Encounters": [
        len(y_train),
        len(y_test)
    ],
    "Readmitted_Within_30_Days": [
        (y_train == 1).sum(),
        (y_test == 1).sum()
    ],
    "Readmission_Rate_%": [
        (y_train == 1).mean() * 100,
        (y_test == 1).mean() * 100
    ]
})

split_target_summary

,Dataset,Total_Encounters,Readmitted_Within_30_Days,Readmission_Rate_%
0,Training,79474,9051,11.388630
1,Testing,19869,2263,11.389602


## 1.1.3 Missing Values & Outliers

- At this stage, we inspect missing values separately in X_train and X_test. We should not split again.

- Also, important correction to my previous wording: we should be careful not to treat ? and NaN as the same thing automatically. We'll inspect the actual values column-by-column first, then decide the appropriate handling.

## Create working copies
# 1.1.3 Missing Values & Outliers Handling
# Step 1: Create working copies


In [13]:
X_train_working = X_train.copy()
X_test_working = X_test.copy()

y_train_working = y_train.copy()
y_test_working = y_test.copy()

X_train_working.shape, X_test_working.shape, y_train_working.shape, y_test_working.shape


((79474, 22), (19869, 22), (79474,), (19869,))

In [14]:
## Check missing values in y_train and y_test


y_missing_summary = pd.DataFrame({
    "Dataset": ["y_train", "y_test"],
    "Missing_Values": [
        y_train_working.isna().sum(),
        y_test_working.isna().sum()
    ]
})

y_missing_summary

,Dataset,Missing_Values
0,y_train,0
1,y_test,0


## dentify missing values in X_train_working
- ? → missing/unavailable categorical information
- actual NaN → numerical/other missing values

In [15]:
# Identify missing values in X_train

missing_train = pd.DataFrame({
    "Question_Mark": [
        (X_train_working[column] == "?").sum()
        if X_train_working[column].dtype == "object"
        else 0
        for column in X_train_working.columns
    ],
    "NaN": X_train_working.isna().sum()
})

missing_train["Total_Missing"] = (
    missing_train["Question_Mark"] +
    missing_train["NaN"]
)

missing_train["Missing_%"] = (
    missing_train["Total_Missing"] /
    len(X_train_working) * 100
)

missing_train.sort_values(
    "Missing_%",
    ascending=False
)

,Question_Mark,NaN,Total_Missing,Missing_%
medical_specialty,38875,0,38875,48.915369
race,1759,0,1759,2.213302
diag_3,1137,0,1137,1.430657
diag_2,286,0,286,0.359866
diag_1,18,0,18,0.022649
diabetesMed,0,0,0,0.000000
insulin,0,0,0,0.000000
num_procedures,0,0,0,0.000000
num_lab_procedures,0,0,0,0.000000
num_medications,0,0,0,0.000000


## Identify Missing Values in X_test_working

In [16]:
#Identify missing values in X_test

missing_test = pd.DataFrame({
    "Question_Mark": [
        (X_test_working[column] == "?").sum()
        if X_test_working[column].dtype == "object"
        else 0
        for column in X_test_working.columns
    ],
    "NaN": X_test_working.isna().sum()
})

missing_test["Total_Missing"] = (
    missing_test["Question_Mark"] +
    missing_test["NaN"]
)

missing_test["Missing_%"] = (
    missing_test["Total_Missing"] /
    len(X_test_working) * 100
)

missing_test.sort_values(
    "Missing_%",
    ascending=False
)

,Question_Mark,NaN,Total_Missing,Missing_%
medical_specialty,9741,0,9741,49.026121
race,475,0,475,2.390659
diag_3,282,0,282,1.419296
diag_2,70,0,70,0.352308
diag_1,2,0,2,0.010066
diabetesMed,0,0,0,0.000000
insulin,0,0,0,0.000000
num_procedures,0,0,0,0.000000
num_lab_procedures,0,0,0,0.000000
num_medications,0,0,0,0.000000


## Handle the missing values

- We will not drop rows. In particular, dropping medical_specialty missing rows would remove almost half of our training data.

- Instead, we will convert ? to an explicit "Missing" category in the working copies only. This is appropriate because these are categorical variables and the fact that information was not recorded can itself be useful to the model.

In [17]:
# Handle categorical missing values

categorical_missing_columns = [
    "race",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3"
]

for column in categorical_missing_columns:
    X_train_working[column] = X_train_working[column].replace(
        "?", "Missing"
    )
    
    X_test_working[column] = X_test_working[column].replace(
        "?", "Missing"
    )

In [18]:
X_train_working.head()

,age,gender,race,admission_type_id,admission_source_id,time_in_hospital,medical_specialty,number_outpatient,number_emergency,number_inpatient,...,diag_2,diag_3,max_glu_serum,A1Cresult,num_medications,num_lab_procedures,num_procedures,insulin,diabetesMed,change
17229,[50-60),Male,AfricanAmerican,1,7,4,InternalMedicine,0,0,0,...,V45,70,None,None,15,49,3,No,No,No
47586,[80-90),Male,Caucasian,2,1,4,Oncology,0,0,0,...,276,153,None,None,15,60,0,Steady,Yes,Ch
49348,[70-80),Female,Caucasian,1,1,7,Missing,2,0,0,...,263,401,None,None,15,60,3,Steady,Yes,Ch
40095,[50-60),Female,AfricanAmerican,3,1,2,Cardiology,0,0,0,...,599,786,None,>8,9,38,0,Down,Yes,Ch
47418,[40-50),Male,AfricanAmerican,1,7,1,InternalMedicine,0,0,0,...,276,276,None,None,15,46,3,Up,Yes,Ch


In [19]:
 X_test_working.head()

,age,gender,race,admission_type_id,admission_source_id,time_in_hospital,medical_specialty,number_outpatient,number_emergency,number_inpatient,...,diag_2,diag_3,max_glu_serum,A1Cresult,num_medications,num_lab_procedures,num_procedures,insulin,diabetesMed,change
48085,[70-80),Male,Caucasian,2,1,2,Urology,0,0,0,...,599,401,None,None,9,16,3,No,No,No
24511,[50-60),Male,Caucasian,3,1,6,Missing,0,0,0,...,250.6,250.53,None,>8,47,56,6,Down,Yes,Ch
55527,[70-80),Female,Caucasian,1,7,4,Missing,0,1,3,...,599,202,None,None,12,66,0,Steady,Yes,No
80599,[70-80),Female,Hispanic,1,7,2,Missing,2,1,1,...,571,786,None,>7,12,37,3,Up,Yes,Ch
28778,[40-50),Male,AfricanAmerican,1,7,4,Psychiatry,3,2,1,...,305,250,None,>8,5,66,5,Down,Yes,Ch


In [20]:
# Verify missing-value handling

missing_after_handling = pd.DataFrame({
    "Train_Missing_?": [
        (X_train_working[column] == "?").sum()
        if X_train_working[column].dtype == "object"
        else 0
        for column in categorical_missing_columns
    ],
    "Test_Missing_?": [
        (X_test_working[column] == "?").sum()
        if X_test_working[column].dtype == "object"
        else 0
        for column in categorical_missing_columns
    ]
}, index=categorical_missing_columns)

missing_after_handling

,Train_Missing_?,Test_Missing_?
race,0,0
medical_specialty,0,0
diag_1,0,0
diag_2,0,0
diag_3,0,0


In [21]:
# Step 6: Verify the Missing category

missing_category_summary = pd.DataFrame({
    "Train_Missing_Category": [
        (X_train_working[column] == "Missing").sum()
        for column in categorical_missing_columns
    ],
    "Test_Missing_Category": [
        (X_test_working[column] == "Missing").sum()
        for column in categorical_missing_columns
    ]
}, index=categorical_missing_columns)

missing_category_summary

,Train_Missing_Category,Test_Missing_Category
race,1759,475
medical_specialty,38875,9741
diag_1,18,2
diag_2,286,70
diag_3,1137,282


## Identify numerical features for outlier assessment

In [22]:
# Step 7: Identify numerical features for outlier assessment

numerical_features = X_train_working.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

numerical_features

['admission_type_id',
 'admission_source_id',
 'time_in_hospital',
 'number_outpatient',
 'number_emergency',
 'number_inpatient',
 'number_diagnoses',
 'num_medications',
 'num_lab_procedures',
 'num_procedures']

In [23]:
numeric_outlier_features = [
    "time_in_hospital",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "num_medications",
    "num_lab_procedures",
    "num_procedures"
]

In [24]:
# Step 8: Numerical outlier assessment

outlier_summary = X_train_working[
    numeric_outlier_features
].describe().T

outlier_summary

,count,mean,std,min,25%,50%,75%,max
time_in_hospital,79474.0,4.383912,2.973032,1.0,2.0,4.0,6.0,14.0
number_outpatient,79474.0,0.372235,1.285251,0.0,0.0,0.0,0.0,42.0
number_emergency,79474.0,0.201324,0.972414,0.0,0.0,0.0,0.0,76.0
number_inpatient,79474.0,0.632848,1.268661,0.0,0.0,0.0,1.0,21.0
number_diagnoses,79474.0,7.407643,1.938118,1.0,6.0,8.0,9.0,16.0
num_medications,79474.0,16.000893,8.113139,1.0,10.0,15.0,20.0,79.0
num_lab_procedures,79474.0,42.881710,19.636173,1.0,31.0,44.0,57.0,129.0
num_procedures,79474.0,1.338375,1.706713,0.0,0.0,1.0,2.0,6.0


- Outlier Handling Conclusion:
Numerical features were assessed for extreme values. The observed high values represent potentially legitimate patient utilization and clinical activity rather than confirmed data errors. Therefore, no global outlier removal or transformation is applied. Outlier handling will remain algorithm-dependent during model-specific preprocessing,

#### ⚙️ 1.1.4 Feature Engineering of x for y


### Feature Generation — Utilization Features

- The first approved feature is prior_utilization, which summarizes the patient's previous outpatient, emergency, and inpatient utilization. This gives the model one overall measure of prior healthcare utilization instead of forcing it to learn the combined pattern only from three separate variables.

In [25]:
##Create feature-engineering copies
# 1.1.4 Feature Engineering
# Step 1: Create feature-engineering copies

X_train_fe = X_train_working.copy()
X_test_fe = X_test_working.copy()

X_train_fe.shape, X_test_fe.shape

((79474, 22), (19869, 22))

In [26]:
# 1.1.4 Feature Engineering
# step 1 Generate prior healthcare utilization feature

X_train_fe["prior_utilization"] = (
    X_train_fe["number_outpatient"]
    + X_train_fe["number_emergency"]
    + X_train_fe["number_inpatient"]
)

X_test_fe["prior_utilization"] = (
    X_test_fe["number_outpatient"]
    + X_test_fe["number_emergency"]
    + X_test_fe["number_inpatient"]
)

X_train_fe[["number_outpatient", "number_emergency", "number_inpatient", "prior_utilization"]].head()

,number_outpatient,number_emergency,number_inpatient,prior_utilization
17229,0,0,0,0
47586,0,0,0,0
49348,2,0,0,2
40095,0,0,0,0
47418,0,0,0,0


## Generate high_inpatient_utilization

- This feature captures whether the patient has a high level of prior inpatient utilization. We use the approved threshold of 2 or more prior inpatient encounters and create a binary indicator (1 = high, 0 = not high).

In [27]:
# Generate high inpatient utilization flag

X_train_fe["high_inpatient_utilization"] = (
    X_train_fe["number_inpatient"] >= 2
).astype(int)

X_test_fe["high_inpatient_utilization"] = (
    X_test_fe["number_inpatient"] >= 2
).astype(int)

X_train_fe[
    ["number_inpatient", "high_inpatient_utilization"]
].head()

,number_inpatient,high_inpatient_utilization
17229,0,0
47586,0,0
49348,0,0
40095,0,0
47418,0,0


## The rule is:

- number_inpatient < 2 → 0
- number_inpatient >= 2 → 1

In [28]:
X_train_fe["high_inpatient_utilization"].value_counts()

high_inpatient_utilization
0    68152
1    11322
Name: count, dtype: int64

In [29]:
pd.crosstab(
    X_train_fe["number_inpatient"],
    X_train_fe["high_inpatient_utilization"]
)

high_inpatient_utilization,0,1
number_inpatient,,
0,52967,0
1,15185,0
2,0,5896
3,0,2576
4,0,1267
5,0,624
6,0,378
7,0,216
8,0,116


## high_emergency_utilization

- We will use the same agreed threshold of 2 or more prior emergency visits:

In [30]:
# Generate high emergency utilization flag

X_train_fe["high_emergency_utilization"] = (
    X_train_fe["number_emergency"] >= 2
).astype(int)

X_test_fe["high_emergency_utilization"] = (
    X_test_fe["number_emergency"] >= 2
).astype(int)

X_train_fe[
    ["number_emergency", "high_emergency_utilization"]
].head()

,number_emergency,high_emergency_utilization
17229,0,0
47586,0,0
49348,0,0
40095,0,0
47418,0,0


In [31]:
X_train_fe["high_emergency_utilization"].value_counts()

high_emergency_utilization
0    76554
1     2920
Name: count, dtype: int64

## diagnosis_complexity

- This feature will use number_diagnoses to represent the patient's diagnostic burden in a simpler category.

- number_diagnoses <= 5  → Low
- number_diagnoses 6–9   → Medium
- number_diagnoses >= 10 → High

In [32]:
# Generate diagnosis complexity category

X_train_fe["diagnosis_complexity"] = pd.cut(
    X_train_fe["number_diagnoses"],
    bins=[-1, 5, 9, np.inf],
    labels=["Low", "Medium", "High"]
)

X_test_fe["diagnosis_complexity"] = pd.cut(
    X_test_fe["number_diagnoses"],
    bins=[-1, 5, 9, np.inf],
    labels=["Low", "Medium", "High"]
)

X_train_fe[
    ["number_diagnoses", "diagnosis_complexity"]
].head()

,number_diagnoses,diagnosis_complexity
17229,9,Medium
47586,7,Medium
49348,9,Medium
40095,6,Medium
47418,9,Medium


In [33]:
X_train_fe["diagnosis_complexity"].value_counts(
    dropna=False
)

diagnosis_complexity
Medium    62809
Low       16581
High         84
Name: count, dtype: int64

## medication_burden_group

- We will convert num_medications into three clinically interpretable groups:

- 1–10   → Low
- 11–20  → Medium
- 21+    → High

In [34]:
# Generate medication burden group

X_train_fe["medication_burden_group"] = pd.cut(
    X_train_fe["num_medications"],
    bins=[0, 10, 20, np.inf],
    labels=["Low", "Medium", "High"]
)

X_test_fe["medication_burden_group"] = pd.cut(
    X_test_fe["num_medications"],
    bins=[0, 10, 20, np.inf],
    labels=["Low", "Medium", "High"]
)

X_train_fe[
    ["num_medications", "medication_burden_group"]
].head()

,num_medications,medication_burden_group
17229,15,Medium
47586,15,Medium
49348,15,Medium
40095,9,Low
47418,15,Medium


In [35]:
X_train_fe["medication_burden_group"].value_counts(
    dropna=False
)

medication_burden_group
Medium    40743
Low       20204
High      18527
Name: count, dtype: int64

## age_group

- We will convert the existing age ranges into broader groups for the readmission model:

Original age	age_group
- [0-10) and [10-20)	Young
- [20-30), [30-40), [40-50)	Adult
- [50-60), [60-70)	Older Adult
- [70-80), [80-90), [90-100)	Elderly

In [36]:
# Generate age group

def create_age_group(age):
    if age in ["[0-10)", "[10-20)"]:
        return "Young"
    elif age in ["[20-30)", "[30-40)", "[40-50)"]:
        return "Adult"
    elif age in ["[50-60)", "[60-70)"]:
        return "Older Adult"
    elif age in ["[70-80)", "[80-90)", "[90-100)"]:
        return "Elderly"
    else:
        return "Missing"

X_train_fe["age_group"] = X_train_fe["age"].apply(
    create_age_group
)

X_test_fe["age_group"] = X_test_fe["age"].apply(
    create_age_group
)

X_train_fe[
    ["age", "age_group"]
].head()

,age,age_group
17229,[50-60),Older Adult
47586,[80-90),Elderly
49348,[70-80),Elderly
40095,[50-60),Older Adult
47418,[40-50),Adult


In [37]:
X_train_fe["age_group"].value_counts(
    dropna=False
)

age_group
Elderly        35542
Older Adult    31245
Adult          12018
Young            669
Name: count, dtype: int64

## glucose_abnormal_flag

- For the approved feature engineering, we will identify whether the patient's maximum serum glucose result is abnormal.

## In this dataset, max_glu_serum uses:

- None → test not performed/documented
- Norm → normal
- >200 → abnormal
- >300 → abnormal

In [38]:
# Generate glucose abnormality flag

X_train_fe["glucose_abnormal_flag"] = (
    X_train_fe["max_glu_serum"]
    .isin([">200", ">300"])
    .astype(int)
)

X_test_fe["glucose_abnormal_flag"] = (
    X_test_fe["max_glu_serum"]
    .isin([">200", ">300"])
    .astype(int)
)

X_train_fe[
    ["max_glu_serum", "glucose_abnormal_flag"]
].head()

,max_glu_serum,glucose_abnormal_flag
17229,None,0
47586,None,0
49348,None,0
40095,None,0
47418,None,0


In [39]:
X_train_fe["glucose_abnormal_flag"].value_counts()

glucose_abnormal_flag
0    77431
1     2043
Name: count, dtype: int64

## insulin_change_flag

- This feature should capture a meaningful treatment-change signal using the available insulin and change columns.

## We will define:

- insulin = not "No" and
- change = "Ch"
→ 1

Otherwise → 0

In [40]:
# Generate insulin treatment change flag

X_train_fe["insulin_change_flag"] = (
    (X_train_fe["insulin"] != "No") &
    (X_train_fe["change"] == "Ch")
).astype(int)

X_test_fe["insulin_change_flag"] = (
    (X_test_fe["insulin"] != "No") &
    (X_test_fe["change"] == "Ch")
).astype(int)

X_train_fe[
    ["insulin", "change", "insulin_change_flag"]
].head()

,insulin,change,insulin_change_flag
17229,No,No,0
47586,Steady,Ch,1
49348,Steady,Ch,1
40095,Down,Ch,1
47418,Up,Ch,1


In [41]:
X_train_fe["insulin_change_flag"].value_counts()

insulin_change_flag
0    49584
1    29890
Name: count, dtype: int64

### Diagnosis Categories
- Because diag_1, diag_2, and diag_3 are ICD-9 diagnosis codes, we'll group them into clinically meaningful categories rather than leaving hundreds of raw codes as separate categories.

In [42]:
# Generate diagnosis category from ICD-9 diagnosis code

def diagnosis_category(code):

    code = str(code).strip()

    # Missing / unavailable diagnosis
    if code in ["?", "Missing", "nan", "None"]:
        return "Missing"

    # Supplementary / external causes
    if code.startswith(("E", "V")):
        return "External_Cause"

    # Diabetes
    try:
        numeric_code = float(code)
    except ValueError:
        return "Other"

    if 250 <= numeric_code < 251:
        return "Diabetes"

    # Infectious diseases
    elif 1 <= numeric_code < 140:
        return "Infectious"

    # Neoplasms
    elif 140 <= numeric_code < 240:
        return "Neoplasms"

    # Endocrine / metabolic
    elif 240 <= numeric_code < 280:
        return "Endocrine_Metabolic"

    # Blood / blood-forming organs
    elif 280 <= numeric_code < 290:
        return "Blood"

    # Mental disorders
    elif 290 <= numeric_code < 320:
        return "Mental"

    # Nervous system
    elif 320 <= numeric_code < 360:
        return "Nervous_System"

    # Circulatory system
    elif 390 <= numeric_code < 460:
        return "Circulatory"

    # Respiratory system
    elif 460 <= numeric_code < 520:
        return "Respiratory"

    # Digestive system
    elif 520 <= numeric_code < 580:
        return "Digestive"

    # Genitourinary system
    elif 580 <= numeric_code < 630:
        return "Genitourinary"

    # Pregnancy
    elif 630 <= numeric_code < 680:
        return "Pregnancy"

    # Skin / subcutaneous tissue
    elif 680 <= numeric_code < 710:
        return "Skin"

    # Musculoskeletal
    elif 710 <= numeric_code < 740:
        return "Musculoskeletal"

    # Congenital anomalies
    elif 740 <= numeric_code < 760:
        return "Congenital"

    # Perinatal conditions
    elif 760 <= numeric_code < 780:
        return "Perinatal"

    # Symptoms / signs / abnormal findings
    elif 780 <= numeric_code < 800:
        return "Symptoms"

    # Injury / poisoning
    elif 800 <= numeric_code < 1000:
        return "Injury"

    else:
        return "Other"


# Apply diagnosis categorization
for column in ["diag_1", "diag_2", "diag_3"]:

    X_train_fe[f"{column}_category"] = (
        X_train_fe[column].apply(diagnosis_category)
    )

    X_test_fe[f"{column}_category"] = (
        X_test_fe[column].apply(diagnosis_category)
    )

X_train_fe[
    [
        "diag_1",
        "diag_1_category",
        "diag_2",
        "diag_2_category",
        "diag_3",
        "diag_3_category"
    ]
].head()

,diag_1,diag_1_category,diag_2,diag_2_category,diag_3,diag_3_category
17229,557,Digestive,V45,External_Cause,70,Infectious
47586,787,Symptoms,276,Endocrine_Metabolic,153,Neoplasms
49348,486,Respiratory,263,Endocrine_Metabolic,401,Circulatory
40095,250.02,Diabetes,599,Genitourinary,786,Symptoms
47418,250.11,Diabetes,276,Endocrine_Metabolic,276,Endocrine_Metabolic


In [43]:
# Verify diagnosis category distributions

diagnosis_category_summary = pd.DataFrame({
    "diag_1_category": X_train_fe["diag_1_category"].value_counts(),
    "diag_2_category": X_train_fe["diag_2_category"].value_counts(),
    "diag_3_category": X_train_fe["diag_3_category"].value_counts()
}).fillna(0).astype(int)

diagnosis_category_summary

,diag_1_category,diag_2_category,diag_3_category
Blood,870,2265,1957
Circulatory,23694,24572,23334
Congenital,44,87,71
Diabetes,6920,10218,13595
Digestive,7208,3110,2787
Endocrine_Metabolic,2127,6395,7159
External_Cause,1267,2013,4017
Genitourinary,3969,6168,4861
Infectious,2014,1477,1450
Injury,5538,1898,1522


In [44]:
## multiple_prior_encounters_flag
# Generate multiple prior encounters flag
# Based only on prior healthcare utilization

X_train_fe["multiple_prior_encounters_flag"] = (
    X_train_fe["prior_utilization"] >= 2
).astype(int)

X_test_fe["multiple_prior_encounters_flag"] = (
    X_test_fe["prior_utilization"] >= 2
).astype(int)

X_train_fe[
    ["prior_utilization", "multiple_prior_encounters_flag"]
].head()

,prior_utilization,multiple_prior_encounters_flag
17229,0,0
47586,0,0
49348,2,1
40095,0,0
47418,0,0


In [45]:
X_train_fe["multiple_prior_encounters_flag"].value_counts()

multiple_prior_encounters_flag
0    59231
1    20243
Name: count, dtype: int64

### high_diagnosis_burden_flag
- number_diagnoses < 10  → 0
- number_diagnoses >= 10 → 1

This complements the diagnosis_complexity category by giving tree/linear models a simple high-burden indicator.

In [46]:
# Generate high diagnosis burden flag

X_train_fe["high_diagnosis_burden_flag"] = (
    X_train_fe["number_diagnoses"] >= 10
).astype(int)

X_test_fe["high_diagnosis_burden_flag"] = (
    X_test_fe["number_diagnoses"] >= 10
).astype(int)

X_train_fe[
    ["number_diagnoses", "high_diagnosis_burden_flag"]
].head()

,number_diagnoses,high_diagnosis_burden_flag
17229,9,0
47586,7,0
49348,9,0
40095,6,0
47418,9,0


In [47]:
X_train_fe["high_diagnosis_burden_flag"].value_counts()

high_diagnosis_burden_flag
0    79390
1       84
Name: count, dtype: int64

In [48]:
# 1.1.4 Feature Engineering
# Final verification of generated features

engineered_features = [
    "prior_utilization",
    "high_inpatient_utilization",
    "high_emergency_utilization",
    "diagnosis_complexity",
    "medication_burden_group",
    "age_group",
    "glucose_abnormal_flag",
    "insulin_change_flag",
    "diag_1_category",
    "diag_2_category",
    "diag_3_category",
    "multiple_prior_encounters_flag",
    "high_diagnosis_burden_flag"
]

feature_engineering_summary = pd.DataFrame({
    "Feature": engineered_features,
    "Train_Present": [
        feature in X_train_fe.columns
        for feature in engineered_features
    ],
    "Test_Present": [
        feature in X_test_fe.columns
        for feature in engineered_features
    ]
})

feature_engineering_summary

,Feature,Train_Present,Test_Present
0,prior_utilization,True,True
1,high_inpatient_utilization,True,True
2,high_emergency_utilization,True,True
3,diagnosis_complexity,True,True
4,medication_burden_group,True,True
5,age_group,True,True
6,glucose_abnormal_flag,True,True
7,insulin_change_flag,True,True
8,diag_1_category,True,True
9,diag_2_category,True,True


In [49]:
pd.DataFrame({
    "Dataset": ["X_train_fe", "X_test_fe"],
    "Rows": [
        len(X_train_fe),
        len(X_test_fe)
    ],
    "Total_Features": [
        X_train_fe.shape[1],
        X_test_fe.shape[1]
    ]
})

,Dataset,Rows,Total_Features
0,X_train_fe,79474,35
1,X_test_fe,19869,35


## 1.1.4 — Feature Selection

- We'll use Mutual Information on the training set only. This measures how much information each feature provides about target_readmit_30d. We will not use X_test or y_test to select features.

## Step 1 — Prepare features for selection

Some columns are categorical even though Pandas may store them as integers, especially:

In [50]:
# 1.1.4 Feature Selection
# Step 1: Prepare training data for feature selection

from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import mutual_info_classif

categorical_features_fs = [
    "age",
    "gender",
    "race",
    "admission_type_id",
    "admission_source_id",
    "medical_specialty",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
    "insulin",
    "diabetesMed",
    "change",
    "diagnosis_complexity",
    "medication_burden_group",
    "age_group",
    "diag_1_category",
    "diag_2_category",
    "diag_3_category"
]

numeric_features_fs = [
    column
    for column in X_train_fe.columns
    if column not in categorical_features_fs
]

# Encode categorical variables only for feature-selection analysis
fs_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_fs = X_train_fe.copy()

X_train_fs[categorical_features_fs] = fs_encoder.fit_transform(
    X_train_fs[categorical_features_fs]
)

# Mutual-information discrete/continuous indicator
discrete_mask_fs = np.array([
    column in categorical_features_fs
    for column in X_train_fs.columns
])

X_train_fs.shape

(79474, 35)

## Step 2 — Calculate Mutual Information

In [51]:
# Step 2: Calculate mutual information scores

mi_scores = mutual_info_classif(
    X_train_fs,
    y_train_working,
    discrete_features=discrete_mask_fs,
    random_state=42
)

feature_selection_summary = pd.DataFrame({
    "Feature": X_train_fe.columns,
    "Mutual_Information": mi_scores
}).sort_values(
    "Mutual_Information",
    ascending=False
).reset_index(drop=True)

feature_selection_summary

,Feature,Mutual_Information
0,number_inpatient,0.011642
1,prior_utilization,0.010114
2,diag_1,0.009490
3,diag_3,0.009256
4,diag_2,0.008089
5,number_diagnoses,0.007594
6,high_inpatient_utilization,0.007009
7,multiple_prior_encounters_flag,0.006861
8,insulin_change_flag,0.004264
9,num_procedures,0.003032


### Feature Selection — What it means

- Feature Selection means choosing the input variables that are most useful for predicting our target:

- target_readmit_30d

- Feature Selection: Mutual Information was used on the training data to rank candidate features according to their individual association with the 30-day readmission target. Features were not removed solely because of low individual scores; redundancy, sparsity, business relevance, and the approved feature-engineering design were also considered. A highly sparse and redundant high_diagnosis_burden_flag was excluded, while clinically relevant features were retained for multivariate model evaluation.

In [52]:
# Feature Selection — Final Classification Features

selected_classification_features = [
    "age",
    "gender",
    "race",
    "admission_type_id",
    "admission_source_id",
    "time_in_hospital",
    "medical_specialty",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "max_glu_serum",
    "A1Cresult",
    "num_medications",
    "num_lab_procedures",
    "num_procedures",
    "insulin",
    "diabetesMed",
    "change",
    "prior_utilization",
    "high_inpatient_utilization",
    "high_emergency_utilization",
    "diagnosis_complexity",
    "medication_burden_group",
    "age_group",
    "glucose_abnormal_flag",
    "insulin_change_flag",
    "diag_1_category",
    "diag_2_category",
    "diag_3_category",
    "multiple_prior_encounters_flag"
]

X_train_selected = X_train_fe[
    selected_classification_features
].copy()

X_test_selected = X_test_fe[
    selected_classification_features
].copy()

X_train_selected.shape, X_test_selected.shape

((79474, 31), (19869, 31))

In [53]:
# Final preprocessing check before encoding

final_check = pd.DataFrame({
    "Dataset": ["X_train_selected", "X_test_selected"],
    "Rows": [
        X_train_selected.shape[0],
        X_test_selected.shape[0]
    ],
    "Features": [
        X_train_selected.shape[1],
        X_test_selected.shape[1]
    ],
    "NaN_Values": [
        X_train_selected.isna().sum().sum(),
        X_test_selected.isna().sum().sum()
    ],
    "Question_Mark_Values": [
        (X_train_selected == "?").sum().sum(),
        (X_test_selected == "?").sum().sum()
    ]
})

final_check

,Dataset,Rows,Features,NaN_Values,Question_Mark_Values
0,X_train_selected,79474,31,0,0
1,X_test_selected,19869,31,0,0


## 1.1.4 Feature Modification
## Step 1 — Encoding

We need to convert every categorical feature to numeric. We will use One-Hot Encoding and keep the result sparse.

In [54]:
categorical_features = [
    "age",
    "gender",
    "race",
    "admission_type_id",
    "admission_source_id",
    "medical_specialty",
    "max_glu_serum",
    "A1Cresult",
    "insulin",
    "diabetesMed",
    "change",
    "diagnosis_complexity",
    "medication_burden_group",
    "age_group",
    "diag_1_category",
    "diag_2_category",
    "diag_3_category"
]

In [55]:
numerical_features = [
    "time_in_hospital",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "num_medications",
    "num_lab_procedures",
    "num_procedures",
    "prior_utilization",
    "high_inpatient_utilization",
    "high_emergency_utilization",
    "glucose_abnormal_flag",
    "insulin_change_flag",
    "multiple_prior_encounters_flag"
]

In [56]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import csr_matrix

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

## Fit on training data and transform both datasets

- Important: We fit the encoder only on training data and use the same encoder to transform test data. This prevents test-set information from influencing preprocessing.

In [57]:
X_train_cat = encoder.fit_transform(
    X_train_selected[categorical_features]
)

X_test_cat = encoder.transform(
    X_test_selected[categorical_features]
)

X_train_num = csr_matrix(
    X_train_selected[numerical_features].astype(np.float32).values
)

X_test_num = csr_matrix(
    X_test_selected[numerical_features].astype(np.float32).values
)

In [58]:
# Combine categorical and numerical features using sparse matrices

from scipy.sparse import hstack

X_train_encoded = hstack(
    [X_train_num, X_train_cat],
    format="csr",
    dtype=np.float32
)

X_test_encoded = hstack(
    [X_test_num, X_test_cat],
    format="csr",
    dtype=np.float32
)

In [59]:
# Verify encoded datasets

encoding_summary = pd.DataFrame({
    "Dataset": [
        "X_train_encoded",
        "X_test_encoded"
    ],
    "Rows": [
        X_train_encoded.shape[0],
        X_test_encoded.shape[0]
    ],
    "Encoded_Features": [
        X_train_encoded.shape[1],
        X_test_encoded.shape[1]
    ],
    "Non_Zero_Values": [
        X_train_encoded.nnz,
        X_test_encoded.nnz
    ]
})

encoding_summary

,Dataset,Rows,Encoded_Features,Non_Zero_Values
0,X_train_encoded,79474,215,1862684
1,X_test_encoded,19869,215,465237


In [60]:
# Feature Modification — Scaling
# Scale pure numerical features for Logistic Regression, KNN and SVM

from sklearn.preprocessing import StandardScaler

numeric_scaler = StandardScaler(
    with_mean=False
)

X_train_num_scaled = numeric_scaler.fit_transform(
    X_train_num
)

X_test_num_scaled = numeric_scaler.transform(
    X_test_num
)

X_train_num_scaled.shape, X_test_num_scaled.shape

((79474, 14), (19869, 14))

In [61]:
### Combine scaled numeric + encoded categorical features
from scipy.sparse import hstack

X_train_scaled = hstack(
    [X_train_num_scaled, X_train_cat],
    format="csr",
    dtype=np.float32
)

X_test_scaled = hstack(
    [X_test_num_scaled, X_test_cat],
    format="csr",
    dtype=np.float32
)

X_train_scaled.shape, X_test_scaled.shape

((79474, 215), (19869, 215))

In [62]:
[name for name in globals() if "encoder" in name.lower() or "scaler" in name.lower()]

['OrdinalEncoder',
 'fs_encoder',
 'OneHotEncoder',
 'encoder',
 'StandardScaler',
 'numeric_scaler']

In [63]:
type(encoder), type(numeric_scaler)

(sklearn.preprocessing._encoders.OneHotEncoder,
 sklearn.preprocessing._data.StandardScaler)

In [64]:
encoder.n_features_in_, numeric_scaler.n_features_in_

(17, 14)

In [65]:
from pathlib import Path
import pickle

pickle_dir = Path(
    r"C:\Users\chinn\OneDrive\Desktop\AI-Hospital-Readmission\ml-fastapi\pickles\model2"
)

pickle_dir.mkdir(parents=True, exist_ok=True)

with open(
    pickle_dir / "classification_onehot_encoder.pkl",
    "wb"
) as file:
    pickle.dump(
        encoder,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )

In [66]:
with open(
    pickle_dir / "classification_scaler.pkl",
    "wb"
) as file:
    pickle.dump(
        numeric_scaler,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )

In [67]:
(
    (pickle_dir / "classification_onehot_encoder.pkl").is_file(),
    (pickle_dir / "classification_scaler.pkl").is_file()
)

(True, True)

In [68]:
import pickle

with open(
    pickle_dir / "classification_onehot_encoder.pkl",
    "rb"
) as file:
    saved_encoder = pickle.load(file)

with open(
    pickle_dir / "classification_scaler.pkl",
    "rb"
) as file:
    saved_scaler = pickle.load(file)

saved_encoder.n_features_in_, saved_scaler.n_features_in_

(17, 14)

In [69]:
X_train_encoded.shape, X_test_encoded.shape

((79474, 215), (19869, 215))

In [70]:
# Verify the final encoded modeling matrices
X_train_encoded.shape, X_test_encoded.shape

((79474, 215), (19869, 215))

In [71]:
# Verify that the final encoded training matrix is preserved
X_train_encoded.shape

(79474, 215)

In [72]:
# Verify that the final scaled matrix is preserved
X_train_scaled.shape, X_test_scaled.shape

((79474, 215), (19869, 215))

In [73]:
# Scale numerical features consistently for LR, KNN and SVM
X_train_scaled = numeric_scaler.transform(
    X_train_selected[numerical_features].to_numpy()
)

X_test_scaled = numeric_scaler.transform(
    X_test_selected[numerical_features].to_numpy()
)

X_train_scaled.shape, X_test_scaled.shape

((79474, 14), (19869, 14))

## 🤖 1.2 Model + Define, Train & Study
- Selecting AI Predictive Model/Algorithm based on y data (Consider all algos applicable when working on project)

## Classification:

- LogR, KNN, NB, SVM, DT, RF, Xgb
- Loading model through python sklearn module
- Training Model with train data

## 1.2.1 Logistic Regression

## Input: X_train_scaled
- Logistic Regression benefits from scaled numerical features and can work directly with our sparse encoded matrix.

In [74]:
#  Logistic Regression
# Define and train the baseline Logistic Regression model

from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train_working
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [75]:
# Recreate the scaled numerical matrices for
# Logistic Regression, KNN and SVM

X_train_scaled = numeric_scaler.transform(
    X_train_selected[numerical_features].to_numpy()
)

X_test_scaled = numeric_scaler.transform(
    X_test_selected[numerical_features].to_numpy()
)

X_train_scaled.shape, X_test_scaled.shape

((79474, 14), (19869, 14))

In [76]:
logistic_model.n_features_in_, X_train_scaled.shape[1], X_test_scaled.shape[1]

(14, 14, 14)

In [77]:
logistic_train_pred = logistic_model.predict(X_train_scaled)

logistic_test_pred = logistic_model.predict(X_test_scaled)

logistic_train_proba = logistic_model.predict_proba(
    X_train_scaled
)[:, 1]

logistic_test_proba = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

In [78]:
# Generate Logistic Regression predictions

logistic_train_pred = logistic_model.predict(
    X_train_scaled
)

logistic_test_pred = logistic_model.predict(
    X_test_scaled
)

logistic_train_proba = logistic_model.predict_proba(
    X_train_scaled
)[:, 1]

logistic_test_proba = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

In [79]:
logistic_model_summary = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Train_Rows": [X_train_scaled.shape[0]],
    "Test_Rows": [X_test_scaled.shape[0]],
    "Encoded_Features": [X_train_scaled.shape[1]],
    "Status": ["Trained Successfully"]
})

logistic_model_summary

,Model,Train_Rows,Test_Rows,Encoded_Features,Status
0,Logistic Regression,79474,19869,14,Trained Successfully


### 1.2.2 KNN Classifier

- KNN needs the scaled matrix, which we already prepared.

In [80]:
# 1.2.2 KNN Classifier
# Define and train baseline KNN

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    n_jobs=-1
)

knn_model.fit(
    X_train_scaled,
    y_train_working
)

,n_neighbors,5
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,-1


In [81]:
# Generate KNN predictions

knn_train_pred = knn_model.predict(
    X_train_scaled
)

knn_test_pred = knn_model.predict(
    X_test_scaled
)

knn_test_proba = knn_model.predict_proba(
    X_test_scaled
)[:, 1]

In [82]:
knn_model_summary = pd.DataFrame({
    "Model": ["KNN Classifier"],
    "Train_Rows": [X_train_scaled.shape[0]],
    "Test_Rows": [X_test_scaled.shape[0]],
    "Encoded_Features": [X_train_scaled.shape[1]],
    "Status": ["Trained Successfully"]
})

knn_model_summary

,Model,Train_Rows,Test_Rows,Encoded_Features,Status
0,KNN Classifier,79474,19869,14,Trained Successfully


## Next: 1.2.3 Naive Bayes

Naive Bayes can use the unscaled encoded sparse representation, so we'll use X_train_encoded / X_test_encoded.

In [83]:
# 1.2.3 Naive Bayes
# Define and train baseline Naive Bayes classifier

from sklearn.naive_bayes import ComplementNB

nb_model = ComplementNB()

nb_model.fit(
    X_train_encoded,
    y_train_working
)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None
,norm,False


In [84]:
nb_model.n_features_in_

215

In [85]:
X_train_encoded.shape, X_test_encoded.shape

((79474, 215), (19869, 215))

In [86]:
# Generate Naive Bayes predictions

nb_train_pred = nb_model.predict(
    X_train_encoded
)

nb_test_pred = nb_model.predict(
    X_test_encoded
)

nb_test_proba = nb_model.predict_proba(
    X_test_encoded
)[:, 1]

In [87]:
nb_model_summary = pd.DataFrame({
    "Model": ["Naive Bayes"],
    "Train_Rows": [X_train_encoded.shape[0]],
    "Test_Rows": [X_test_encoded.shape[0]],
    "Encoded_Features": [X_train_encoded.shape[1]],
    "Status": ["Trained Successfully"]
})

nb_model_summary

,Model,Train_Rows,Test_Rows,Encoded_Features,Status
0,Naive Bayes,79474,19869,215,Trained Successfully


## 1.2.4 SVM Classifier

- Because our dataset has 79,474 rows and 215 encoded features, we should use LinearSVC rather than a kernel SVC. It is much more practical for this high-dimensional sparse dataset.

In [88]:
# 1.2.4 SVM Classifier
# Define and train baseline Linear SVM classifier

from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    dual=False,
    max_iter=3000,
    random_state=42
)

svm_model.fit(
    X_train_scaled,
    y_train_working
)

,penalty,'l2'
,loss,'squared_hinge'
,dual,False
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,42


- Generate predictions and decision scores

LinearSVC does not provide predict_proba(), so for the baseline we use its decision score. This score can still be used later for ROC-AUC and PR-AUC.

In [89]:
# Generate SVM predictions and decision scores

svm_train_pred = svm_model.predict(
    X_train_scaled
)

svm_test_pred = svm_model.predict(
    X_test_scaled
)

svm_train_score = svm_model.decision_function(
    X_train_scaled
)

svm_test_score = svm_model.decision_function(
    X_test_scaled
)

In [90]:
svm_model_summary = pd.DataFrame({
    "Model": ["SVM Classifier"],
    "Train_Rows": [X_train_scaled.shape[0]],
    "Test_Rows": [X_test_scaled.shape[0]],
    "Encoded_Features": [X_train_scaled.shape[1]],
    "Status": ["Trained Successfully"]
})

svm_model_summary

,Model,Train_Rows,Test_Rows,Encoded_Features,Status
0,SVM Classifier,79474,19869,14,Trained Successfully


## 1.2.5 Decision Tree Classifier

- Decision Tree does not require scaling, so use:

- X_train_encoded
- X_test_encoded

In [91]:
# 1.2.5 Decision Tree Classifier
# Define and train baseline Decision Tree

from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

dt_model.fit(
    X_train_encoded,
    y_train_working
)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,'balanced'


In [92]:
# Generate Decision Tree predictions

dt_train_pred = dt_model.predict(
    X_train_encoded
)

dt_test_pred = dt_model.predict(
    X_test_encoded
)

dt_train_proba = dt_model.predict_proba(
    X_train_encoded
)[:, 1]

dt_test_proba = dt_model.predict_proba(
    X_test_encoded
)[:, 1]

In [93]:
dt_model_summary = pd.DataFrame({
    "Model": ["Decision Tree Classifier"],
    "Train_Rows": [X_train_encoded.shape[0]],
    "Test_Rows": [X_test_encoded.shape[0]],
    "Encoded_Features": [X_train_encoded.shape[1]],
    "Status": ["Trained Successfully"]
})

dt_model_summary

,Model,Train_Rows,Test_Rows,Encoded_Features,Status
0,Decision Tree Classifier,79474,19869,215,Trained Successfully


## 1.2.6 Random Forest Classifier

- Random Forest is tree-based, so it does not require scaling. Use the encoded sparse matrix.

-  Define and train

In [94]:
# 1.2.6 Random Forest Classifier
# Define and train baseline Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(
    X_train_encoded,
    y_train_working
)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [95]:
# Generate Random Forest predictions

rf_train_pred = rf_model.predict(
    X_train_encoded
)

rf_test_pred = rf_model.predict(
    X_test_encoded
)

rf_train_proba = rf_model.predict_proba(
    X_train_encoded
)[:, 1]

rf_test_proba = rf_model.predict_proba(
    X_test_encoded
)[:, 1]

In [96]:
rf_model_summary = pd.DataFrame({
    "Model": ["Random Forest Classifier"],
    "Train_Rows": [X_train_encoded.shape[0]],
    "Test_Rows": [X_test_encoded.shape[0]],
    "Encoded_Features": [X_train_encoded.shape[1]],
    "Status": ["Trained Successfully"]
})

rf_model_summary

,Model,Train_Rows,Test_Rows,Encoded_Features,Status
0,Random Forest Classifier,79474,19869,215,Trained Successfully


## 1.2.7 XGBoost Classifier

- We'll use the unscaled sparse encoded matrix:

- X_train_encoded
- X_test_encoded

In [97]:
# 1.2.7 XGBoost Classifier
# Define and train baseline XGBoost

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_encoded,
    y_train_working
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'logloss'


In [98]:
# Generate XGBoost predictions

xgb_train_pred = xgb_model.predict(
    X_train_encoded
)

xgb_test_pred = xgb_model.predict(
    X_test_encoded
)

xgb_train_proba = xgb_model.predict_proba(
    X_train_encoded
)[:, 1]

xgb_test_proba = xgb_model.predict_proba(
    X_test_encoded
)[:, 1]

In [99]:
xgb_model_summary = pd.DataFrame({
    "Model": ["XGBoost Classifier"],
    "Train_Rows": [X_train_encoded.shape[0]],
    "Test_Rows": [X_test_encoded.shape[0]],
    "Encoded_Features": [X_train_encoded.shape[1]],
    "Status": ["Trained Successfully"]
})

xgb_model_summary

,Model,Train_Rows,Test_Rows,Encoded_Features,Status
0,XGBoost Classifier,79474,19869,215,Trained Successfully


## Calculate Train/Test F1

In [100]:
from sklearn.metrics import f1_score

baseline_classification_results = []

model_predictions = {
    "Logistic Regression": (
        logistic_train_pred,
        logistic_test_pred
    ),
    "KNN": (
        knn_train_pred,
        knn_test_pred
    ),
    "Naive Bayes": (
        nb_train_pred,
        nb_test_pred
    ),
    "SVM": (
        svm_train_pred,
        svm_test_pred
    ),
    "Decision Tree": (
        dt_train_pred,
        dt_test_pred
    ),
    "Random Forest": (
        rf_train_pred,
        rf_test_pred
    ),
    "XGBoost": (
        xgb_train_pred,
        xgb_test_pred
    )
}

for model_name, (train_pred, test_pred) in model_predictions.items():

    train_f1 = f1_score(
        y_train_working,
        train_pred
    )

    test_f1 = f1_score(
        y_test_working,
        test_pred
    )

    f1_gap = train_f1 - test_f1

    baseline_classification_results.append({
        "Model": model_name,
        "Train_F1": train_f1,
        "Test_F1": test_f1,
        "F1_Gap": f1_gap
    })

baseline_classification_results = pd.DataFrame(
    baseline_classification_results
)

baseline_classification_results.sort_values(
    "Test_F1",
    ascending=False
).reset_index(drop=True)

,Model,Train_F1,Test_F1,F1_Gap
0,SVM,0.252982,0.254157,-0.001175
1,Logistic Regression,0.253439,0.253872,-0.000433
2,Naive Bayes,0.251239,0.252929,-0.001690
3,Decision Tree,1.000000,0.159950,0.840050
4,KNN,0.966693,0.087601,0.879092
5,XGBoost,0.069955,0.027539,0.042417
6,Random Forest,1.000000,0.019189,0.980811


## Bias-Variance Fit

- We will classify the fit from the train/test F1 relationship, rather than blindly calling every high-scoring model "Good Fit".

In [101]:
def classify_fit(train_f1, test_f1, gap):

    if train_f1 < 0.50 and test_f1 < 0.50:
        return "Underfitting"

    elif gap > 0.10:
        return "Overfitting"

    else:
        return "Good Fit"


baseline_classification_results["Bias_Variance_Fit"] = (
    baseline_classification_results.apply(
        lambda row: classify_fit(
            row["Train_F1"],
            row["Test_F1"],
            row["F1_Gap"]
        ),
        axis=1
    )
)

baseline_classification_results

,Model,Train_F1,Test_F1,F1_Gap,Bias_Variance_Fit
0,Logistic Regression,0.253439,0.253872,-0.000433,Underfitting
1,KNN,0.966693,0.087601,0.879092,Overfitting
2,Naive Bayes,0.251239,0.252929,-0.001690,Underfitting
3,SVM,0.252982,0.254157,-0.001175,Underfitting
4,Decision Tree,1.000000,0.159950,0.840050,Overfitting
5,Random Forest,1.000000,0.019189,0.980811,Overfitting
6,XGBoost,0.069955,0.027539,0.042417,Underfitting


## Cross-Validation F1

- For the CV comparison, use 3-fold Stratified CV so the minority-class proportion is preserved.

In [102]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

cv_models = {
    "Logistic Regression": (
        logistic_model,
        X_train_scaled
    ),
    "Naive Bayes": (
        nb_model,
        X_train_encoded
    ),
    "SVM": (
        svm_model,
        X_train_scaled
    ),
    "Decision Tree": (
        dt_model,
        X_train_encoded
    ),
    "Random Forest": (
        rf_model,
        X_train_encoded
    ),
    "XGBoost": (
        xgb_model,
        X_train_encoded
    )
}

cv_results = []

for model_name, (model, X_data) in cv_models.items():

    scores = cross_val_score(
        model,
        X_data,
        y_train_working,
        cv=cv_strategy,
        scoring="f1",
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "CV_F1_Mean": scores.mean(),
        "CV_F1_STD": scores.std()
    })

cv_results = pd.DataFrame(cv_results)
cv_results

,Model,CV_F1_Mean,CV_F1_STD
0,Logistic Regression,0.253533,0.001899
1,Naive Bayes,0.249893,0.002770
2,SVM,0.252206,0.002466
3,Decision Tree,0.157604,0.002868
4,Random Forest,0.010075,0.000820
5,XGBoost,0.028026,0.002011


In [103]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

detailed_results = []

# Probability/score outputs
model_scores = {
    "Logistic Regression": logistic_test_proba,
    "KNN": knn_test_proba,
    "Naive Bayes": nb_test_proba,
    "SVM": svm_test_score,
    "Decision Tree": dt_test_proba,
    "Random Forest": rf_test_proba,
    "XGBoost": xgb_test_proba
}

for model_name, (train_pred, test_pred) in model_predictions.items():

    score = model_scores[model_name]

    detailed_results.append({
        "Model": model_name,
        "Precision": precision_score(
            y_test_working,
            test_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_working,
            test_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test_working,
            test_pred,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test_working,
            score
        ),
        "PR_AUC": average_precision_score(
            y_test_working,
            score
        ),
        "Brier_Score": (
            brier_score_loss(
                y_test_working,
                score
            )
            if model_name != "SVM"
            else np.nan
        )
    })

detailed_results = pd.DataFrame(
    detailed_results
).sort_values(
    "PR_AUC",
    ascending=False
).reset_index(drop=True)

detailed_results

,Model,Precision,Recall,F1,ROC_AUC,PR_AUC,Brier_Score
0,XGBoost,0.524590,0.014141,0.027539,0.647785,0.204967,0.097526
1,Logistic Regression,0.174804,0.463544,0.253872,0.635626,0.202601,0.232465
2,SVM,0.176704,0.452497,0.254157,0.634109,0.200974,NaN
3,Naive Bayes,0.181906,0.414936,0.252929,0.631174,0.195782,0.229279
4,Random Forest,0.733333,0.009722,0.019189,0.619486,0.195200,0.098471
5,KNN,0.184397,0.057446,0.087601,0.549992,0.133935,0.123209
6,Decision Tree,0.150921,0.170128,0.159950,0.523551,0.120195,0.203533


In [104]:
# KNN Cross-Validation

knn_cv_scores = cross_val_score(
    knn_model,
    X_train_scaled,
    y_train_working,
    cv=cv_strategy,
    scoring="f1",
    n_jobs=-1,
    pre_dispatch="2*n_jobs"
)

knn_cv_result = pd.DataFrame({
    "Model": ["KNN"],
    "CV_F1_Mean": [knn_cv_scores.mean()],
    "CV_F1_STD": [knn_cv_scores.std()]
})

knn_cv_result

,Model,CV_F1_Mean,CV_F1_STD
0,KNN,0.084609,0.003711


In [105]:
# Combine baseline F1 and Cross-Validation results

classification_model_comparison = (
    baseline_classification_results
    .merge(
        pd.concat(
            [
                cv_results,
                knn_cv_result
            ],
            ignore_index=True
        ),
        on="Model",
        how="left"
    )
)

classification_model_comparison[
    [
        "Model",
        "Train_F1",
        "Test_F1",
        "F1_Gap",
        "Bias_Variance_Fit",
        "CV_F1_Mean",
        "CV_F1_STD"
    ]
].sort_values(
    "Test_F1",
    ascending=False
).reset_index(drop=True)

,Model,Train_F1,Test_F1,F1_Gap,Bias_Variance_Fit,CV_F1_Mean,CV_F1_STD
0,SVM,0.252982,0.254157,-0.001175,Underfitting,0.252206,0.002466
1,Logistic Regression,0.253439,0.253872,-0.000433,Underfitting,0.253533,0.001899
2,Naive Bayes,0.251239,0.252929,-0.001690,Underfitting,0.249893,0.002770
3,Decision Tree,1.000000,0.159950,0.840050,Overfitting,0.157604,0.002868
4,KNN,0.966693,0.087601,0.879092,Overfitting,0.084609,0.003711
5,XGBoost,0.069955,0.027539,0.042417,Underfitting,0.028026,0.002011
6,Random Forest,1.000000,0.019189,0.980811,Overfitting,0.010075,0.000820


## 1.3.1 Detailed Evaluation — All Models

In [106]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Test predictions and scores/probabilities
test_predictions = {
    "Logistic Regression": logistic_test_pred,
    "KNN": knn_test_pred,
    "Naive Bayes": nb_test_pred,
    "SVM": svm_test_pred,
    "Decision Tree": dt_test_pred,
    "Random Forest": rf_test_pred,
    "XGBoost": xgb_test_pred
}

test_scores = {
    "Logistic Regression": logistic_test_proba,
    "KNN": knn_test_proba,
    "Naive Bayes": nb_test_proba,
    "SVM": svm_test_score,
    "Decision Tree": dt_test_proba,
    "Random Forest": rf_test_proba,
    "XGBoost": xgb_test_proba
}

# Models with true probability estimates
probability_models = {
    "Logistic Regression": logistic_test_proba,
    "KNN": knn_test_proba,
    "Naive Bayes": nb_test_proba,
    "Decision Tree": dt_test_proba,
    "Random Forest": rf_test_proba,
    "XGBoost": xgb_test_proba
}

detailed_evaluation = []

for model_name in test_predictions:

    predictions = test_predictions[model_name]
    scores = test_scores[model_name]

    result = {
        "Model": model_name,
        "Precision": precision_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test_working,
            scores
        ),
        "PR_AUC": average_precision_score(
            y_test_working,
            scores
        )
    }

    # Brier score only where real probability estimates exist
    if model_name in probability_models:
        result["Brier_Score"] = brier_score_loss(
            y_test_working,
            probability_models[model_name]
        )
    else:
        result["Brier_Score"] = np.nan

    detailed_evaluation.append(result)

detailed_evaluation = pd.DataFrame(
    detailed_evaluation
)

detailed_evaluation.sort_values(
    "PR_AUC",
    ascending=False
).reset_index(drop=True)

,Model,Precision,Recall,F1,ROC_AUC,PR_AUC,Brier_Score
0,XGBoost,0.524590,0.014141,0.027539,0.647785,0.204967,0.097526
1,Logistic Regression,0.174804,0.463544,0.253872,0.635626,0.202601,0.232465
2,SVM,0.176704,0.452497,0.254157,0.634109,0.200974,NaN
3,Naive Bayes,0.181906,0.414936,0.252929,0.631174,0.195782,0.229279
4,Random Forest,0.733333,0.009722,0.019189,0.619486,0.195200,0.098471
5,KNN,0.184397,0.057446,0.087601,0.549992,0.133935,0.123209
6,Decision Tree,0.150921,0.170128,0.159950,0.523551,0.120195,0.203533


In [107]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_svm = CalibratedClassifierCV(
    svm_model,
    method="sigmoid",
    cv=3
)

calibrated_svm.fit(
    X_train_scaled,
    y_train_working
)

,estimator,LinearSVC(cla...ndom_state=42)
,method,'sigmoid'
,cv,3
,n_jobs,None
,ensemble,'auto'
,penalty,'l2'
,loss,'squared_hinge'
,dual,False
,tol,0.0001
,C,1.0
,multi_class,'ovr'


In [108]:
svm_calibrated_proba = calibrated_svm.predict_proba(
    X_test_scaled
)[:, 1]

svm_brier_score = brier_score_loss(
    y_test_working,
    svm_calibrated_proba
)

In [109]:
from sklearn.metrics import confusion_matrix
# 1.3.2 Confusion Matrix — Baseline Models

confusion_matrices = {}

for model_name, predictions in test_predictions.items():
    confusion_matrices[model_name] = confusion_matrix(
        y_test_working,
        predictions
    )

for model_name, cm in confusion_matrices.items():
    print(f"\n{model_name}")
    print(cm)


Logistic Regression
[[12654  4952]
 [ 1214  1049]]

KNN
[[17031   575]
 [ 2133   130]]

Naive Bayes
[[13383  4223]
 [ 1324   939]]

SVM
[[12835  4771]
 [ 1239  1024]]

Decision Tree
[[15440  2166]
 [ 1878   385]]

Random Forest
[[17598     8]
 [ 2241    22]]

XGBoost
[[17577    29]
 [ 2231    32]]


In [110]:
# Confusion Matrix Summary

confusion_matrix_summary = []

for model_name, cm in confusion_matrices.items():

    tn, fp, fn, tp = cm.ravel()

    confusion_matrix_summary.append({
        "Model": model_name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

confusion_matrix_summary = pd.DataFrame(
    confusion_matrix_summary
)

confusion_matrix_summary

,Model,TN,FP,FN,TP
0,Logistic Regression,12654,4952,1214,1049
1,KNN,17031,575,2133,130
2,Naive Bayes,13383,4223,1324,939
3,SVM,12835,4771,1239,1024
4,Decision Tree,15440,2166,1878,385
5,Random Forest,17598,8,2241,22
6,XGBoost,17577,29,2231,32


##  Calibration / Brier Score

- our problem statement explicitly requires Calibration / Brier score, so we should complete that before tuning.

- SVM currently has NaN because LinearSVC doesn't produce probabilities. We'll calibrate a copy of the SVM and then calculate its Brier score.

## Step 1 — Calibrate SVM

In [111]:
# 1.3.3 SVM Probability Calibration

from sklearn.calibration import CalibratedClassifierCV

calibrated_svm = CalibratedClassifierCV(
    estimator=svm_model,
    method="sigmoid",
    cv=3,
    n_jobs=-1
)

calibrated_svm.fit(
    X_train_scaled,
    y_train_working
)

,estimator,LinearSVC(cla...ndom_state=42)
,method,'sigmoid'
,cv,3
,n_jobs,-1
,ensemble,'auto'
,penalty,'l2'
,loss,'squared_hinge'
,dual,False
,tol,0.0001
,C,1.0
,multi_class,'ovr'


In [112]:
# Generate calibrated SVM probabilities

svm_calibrated_proba = calibrated_svm.predict_proba(
    X_test_scaled
)[:, 1]

svm_calibrated_pred = (
    svm_calibrated_proba >= 0.5
).astype(int)

In [113]:
# Calculate calibrated SVM Brier score

svm_calibrated_brier = brier_score_loss(
    y_test_working,
    svm_calibrated_proba
)

pd.DataFrame({
    "Model": ["Calibrated SVM"],
    "Brier_Score": [svm_calibrated_brier]
})

,Model,Brier_Score
0,Calibrated SVM,0.097808


In [114]:
# Final baseline comparison table

final_baseline_classification = (
    classification_model_comparison[
        [
            "Model",
            "Train_F1",
            "Test_F1",
            "F1_Gap",
            "Bias_Variance_Fit",
            "CV_F1_Mean",
            "CV_F1_STD"
        ]
    ]
    .merge(
        detailed_evaluation[
            [
                "Model",
                "Precision",
                "Recall",
                "F1",
                "ROC_AUC",
                "PR_AUC",
                "Brier_Score"
            ]
        ],
        on="Model",
        how="left"
    )
)

# Replace SVM Brier NaN with calibrated Brier score
final_baseline_classification.loc[
    final_baseline_classification["Model"] == "SVM",
    "Brier_Score"
] = svm_calibrated_brier

final_baseline_classification = (
    final_baseline_classification
    .sort_values("PR_AUC", ascending=False)
    .reset_index(drop=True)
)

final_baseline_classification

,Model,Train_F1,Test_F1,F1_Gap,Bias_Variance_Fit,CV_F1_Mean,CV_F1_STD,Precision,Recall,F1,ROC_AUC,PR_AUC,Brier_Score
0,XGBoost,0.069955,0.027539,0.042417,Underfitting,0.028026,0.002011,0.524590,0.014141,0.027539,0.647785,0.204967,0.097526
1,Logistic Regression,0.253439,0.253872,-0.000433,Underfitting,0.253533,0.001899,0.174804,0.463544,0.253872,0.635626,0.202601,0.232465
2,SVM,0.252982,0.254157,-0.001175,Underfitting,0.252206,0.002466,0.176704,0.452497,0.254157,0.634109,0.200974,0.097808
3,Naive Bayes,0.251239,0.252929,-0.001690,Underfitting,0.249893,0.002770,0.181906,0.414936,0.252929,0.631174,0.195782,0.229279
4,Random Forest,1.000000,0.019189,0.980811,Overfitting,0.010075,0.000820,0.733333,0.009722,0.019189,0.619486,0.195200,0.098471
5,KNN,0.966693,0.087601,0.879092,Overfitting,0.084609,0.003711,0.184397,0.057446,0.087601,0.549992,0.133935,0.123209
6,Decision Tree,1.000000,0.159950,0.840050,Overfitting,0.157604,0.002868,0.150921,0.170128,0.159950,0.523551,0.120195,0.203533


## ✨ 1.4 Hyperparameter Tuning

-Improving Model/Best Model Performance further by Hyperparameter Tuning (if required)

## 1.4.1 Logistic Regression — Hyperparameter Tuning

We will optimize F1-score, because our positive class is only about 11.39% and F1 balances precision and recall.

In [115]:
# 1.4.1 Logistic Regression Hyperparameter Tuning

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression

lr_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

lr_param_grid = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "class_weight": [None, "balanced"]
}

lr_grid_search = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=1000,
        solver="liblinear",
        random_state=42
    ),
    param_grid=lr_param_grid,
    scoring="f1",
    cv=lr_cv,
    n_jobs=-1,
    refit=True
)

lr_grid_search.fit(
    X_train_scaled,
    y_train_working
)

,estimator,LogisticRegre...r='liblinear')
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [116]:
# Best Logistic Regression parameters

best_lr_params = lr_grid_search.best_params_
best_lr_cv_f1 = lr_grid_search.best_score_

pd.DataFrame({
    "Best_C": [best_lr_params["C"]],
    "Best_Class_Weight": [best_lr_params["class_weight"]],
    "Best_CV_F1": [best_lr_cv_f1]
})

,Best_C,Best_Class_Weight,Best_CV_F1
0,0.01,balanced,0.2537


In [117]:
# Logistic Regression tuning results

lr_tuning_results = pd.DataFrame(
    lr_grid_search.cv_results_
)

lr_tuning_results[
    [
        "param_C",
        "param_class_weight",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
).reset_index(drop=True)

,param_C,param_class_weight,mean_test_score,std_test_score,rank_test_score
0,0.01,balanced,0.253700,0.001720,1
1,10.00,balanced,0.253594,0.001835,2
2,1.00,balanced,0.253542,0.001862,3
3,0.10,balanced,0.253446,0.001850,4
4,10.00,None,0.024385,0.001659,5
5,1.00,None,0.024382,0.001657,6
6,0.10,None,0.024172,0.001586,7
7,0.01,None,0.020374,0.000841,8


In [118]:
# 1.4.2 KNN Hyperparameter Tuning

from sklearn.model_selection import GridSearchCV

knn_param_grid = {
    "n_neighbors": [11, 21, 31],
    "weights": ["uniform", "distance"]
}

knn_grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(
        n_jobs=-1
    ),
    param_grid=knn_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    pre_dispatch="2*n_jobs",
    refit=True
)

knn_grid_search.fit(
    X_train_scaled,
    y_train_working
)

,estimator,KNeighborsCla...ier(n_jobs=-1)
,param_grid,"{'n_neighbors': [11, 21, ...], 'weights': ['uniform', 'distance']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,11


In [119]:
# Best KNN parameters

best_knn_params = knn_grid_search.best_params_
best_knn_cv_f1 = knn_grid_search.best_score_

pd.DataFrame({
    "Best_n_neighbors": [best_knn_params["n_neighbors"]],
    "Best_weights": [best_knn_params["weights"]],
    "Best_CV_F1": [best_knn_cv_f1]
})

,Best_n_neighbors,Best_weights,Best_CV_F1
0,11,distance,0.04643


In [120]:
# KNN tuning results

knn_tuning_results = pd.DataFrame(
    knn_grid_search.cv_results_
)

knn_tuning_results[
    [
        "param_n_neighbors",
        "param_weights",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
).reset_index(drop=True)

,param_n_neighbors,param_weights,mean_test_score,std_test_score,rank_test_score
0,11,distance,0.046430,0.001571,1
1,11,uniform,0.033317,0.003407,2
2,21,distance,0.029592,0.001005,3
3,31,distance,0.027977,0.002376,4
4,21,uniform,0.020519,0.000318,5
5,31,uniform,0.020195,0.001055,6


In [121]:
# Create the best tuned KNN model

tuned_knn_model = knn_grid_search.best_estimator_

# Predictions
tuned_knn_train_pred = tuned_knn_model.predict(X_train_scaled)
tuned_knn_test_pred = tuned_knn_model.predict(X_test_scaled)

# Probabilities
tuned_knn_train_proba = tuned_knn_model.predict_proba(
    X_train_scaled
)[:, 1]

tuned_knn_test_proba = tuned_knn_model.predict_proba(
    X_test_scaled
)[:, 1]

# Calculate F1
tuned_knn_train_f1 = f1_score(
    y_train_working,
    tuned_knn_train_pred,
    zero_division=0
)

tuned_knn_test_f1 = f1_score(
    y_test_working,
    tuned_knn_test_pred,
    zero_division=0
)

tuned_knn_f1_gap = (
    tuned_knn_train_f1 - tuned_knn_test_f1
)

pd.DataFrame({
    "Model": ["Tuned KNN"],
    "Train_F1": [tuned_knn_train_f1],
    "Test_F1": [tuned_knn_test_f1],
    "F1_Gap": [tuned_knn_f1_gap],
    "Best_CV_F1": [knn_grid_search.best_score_]
})

,Model,Train_F1,Test_F1,F1_Gap,Best_CV_F1
0,Tuned KNN,0.966693,0.049645,0.917048,0.04643


In [122]:
pd.DataFrame({
    "Best_n_neighbors": [
        knn_grid_search.best_params_["n_neighbors"]
    ],
    "Best_weights": [
        knn_grid_search.best_params_["weights"]
    ],
    "Best_CV_F1": [
        knn_grid_search.best_score_
    ]
})

,Best_n_neighbors,Best_weights,Best_CV_F1
0,11,distance,0.04643


##  1.4.3 SVM Hyperparameter Tuning

- SVM is a much better candidate because the baseline was stable:

- Baseline Test F1 = 0.259457
- Baseline CV F1   = 0.256783



In [123]:
# 1.4.3 SVM Hyperparameter Tuning

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

svm_param_grid = {
    "C": [0.01, 0.1, 1.0, 10.0],
    "class_weight": [None, "balanced"]
}

svm_grid_search = GridSearchCV(
    estimator=LinearSVC(
        dual=False,
        max_iter=3000,
        random_state=42
    ),
    param_grid=svm_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    pre_dispatch="2*n_jobs",
    refit=True
)

svm_grid_search.fit(
    X_train_scaled,
    y_train_working
)

,estimator,LinearSVC(dua...ndom_state=42)
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [124]:
# Best SVM parameters

best_svm_params = svm_grid_search.best_params_
best_svm_cv_f1 = svm_grid_search.best_score_

pd.DataFrame({
    "Best_C": [best_svm_params["C"]],
    "Best_Class_Weight": [best_svm_params["class_weight"]],
    "Best_CV_F1": [best_svm_cv_f1]
})

,Best_C,Best_Class_Weight,Best_CV_F1
0,0.01,balanced,0.25229


In [125]:
# SVM tuning results

svm_tuning_results = pd.DataFrame(
    svm_grid_search.cv_results_
)

svm_tuning_results[
    [
        "param_C",
        "param_class_weight",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
).reset_index(drop=True)

,param_C,param_class_weight,mean_test_score,std_test_score,rank_test_score
0,0.01,balanced,0.252290,0.002522,1
1,1.00,balanced,0.252206,0.002466,2
2,10.00,balanced,0.252206,0.002466,2
3,0.10,balanced,0.252132,0.002558,4
4,0.01,None,0.014180,0.001115,5
5,0.10,None,0.014180,0.001115,5
6,1.00,None,0.014180,0.001115,5
7,10.00,None,0.014180,0.001115,5


In [126]:
# Tuned SVM

tuned_svm_model = svm_grid_search.best_estimator_

tuned_svm_train_pred = tuned_svm_model.predict(
    X_train_scaled
)

tuned_svm_test_pred = tuned_svm_model.predict(
    X_test_scaled
)

tuned_svm_train_score = tuned_svm_model.decision_function(
    X_train_scaled
)

tuned_svm_test_score = tuned_svm_model.decision_function(
    X_test_scaled
)

tuned_svm_train_f1 = f1_score(
    y_train_working,
    tuned_svm_train_pred,
    zero_division=0
)

tuned_svm_test_f1 = f1_score(
    y_test_working,
    tuned_svm_test_pred,
    zero_division=0
)

pd.DataFrame({
    "Model": ["Tuned SVM"],
    "Train_F1": [tuned_svm_train_f1],
    "Test_F1": [tuned_svm_test_f1],
    "F1_Gap": [
        tuned_svm_train_f1 - tuned_svm_test_f1
    ],
    "Best_CV_F1": [svm_grid_search.best_score_]
})

,Model,Train_F1,Test_F1,F1_Gap,Best_CV_F1
0,Tuned SVM,0.253095,0.254509,-0.001414,0.25229


In [127]:
# 1.4.4 Naive Bayes Hyperparameter Tuning

from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import GridSearchCV

nb_param_grid = {
    "alpha": [0.01, 0.1, 1.0, 10.0]
}

nb_grid_search = GridSearchCV(
    estimator=ComplementNB(),
    param_grid=nb_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    refit=True
)

nb_grid_search.fit(
    X_train_encoded,
    y_train_working
)

,estimator,ComplementNB()
,param_grid,"{'alpha': [0.01, 0.1, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,1.0


In [128]:
# Best Naive Bayes parameters

best_nb_params = nb_grid_search.best_params_
best_nb_cv_f1 = nb_grid_search.best_score_

pd.DataFrame({
    "Best_Alpha": [best_nb_params["alpha"]],
    "Best_CV_F1": [best_nb_cv_f1]
})

,Best_Alpha,Best_CV_F1
0,1.0,0.249893


In [129]:
# Naive Bayes tuning results

nb_tuning_results = pd.DataFrame(
    nb_grid_search.cv_results_
)

nb_tuning_results[
    [
        "param_alpha",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
).reset_index(drop=True)

,param_alpha,mean_test_score,std_test_score,rank_test_score
0,1.00,0.249893,0.002770,1
1,0.10,0.249560,0.003180,2
2,10.00,0.249542,0.002884,3
3,0.01,0.249435,0.003237,4


In [130]:
# Tuned Naive Bayes

tuned_nb_model = nb_grid_search.best_estimator_

tuned_nb_train_pred = tuned_nb_model.predict(
    X_train_encoded
)

tuned_nb_test_pred = tuned_nb_model.predict(
    X_test_encoded
)

tuned_nb_train_proba = tuned_nb_model.predict_proba(
    X_train_encoded
)[:, 1]

tuned_nb_test_proba = tuned_nb_model.predict_proba(
    X_test_encoded
)[:, 1]

tuned_nb_train_f1 = f1_score(
    y_train_working,
    tuned_nb_train_pred,
    zero_division=0
)

tuned_nb_test_f1 = f1_score(
    y_test_working,
    tuned_nb_test_pred,
    zero_division=0
)

pd.DataFrame({
    "Model": ["Tuned Naive Bayes"],
    "Train_F1": [tuned_nb_train_f1],
    "Test_F1": [tuned_nb_test_f1],
    "F1_Gap": [
        tuned_nb_train_f1 - tuned_nb_test_f1
    ],
    "Best_CV_F1": [nb_grid_search.best_score_]
})

,Model,Train_F1,Test_F1,F1_Gap,Best_CV_F1
0,Tuned Naive Bayes,0.251239,0.252929,-0.00169,0.249893


## 1.4.5 Decision Tree Hyperparameter Tuning

- The baseline Decision Tree had:

- Train F1 = 1.000000
- Test F1  = 0.159950
- CV F1    = 0.157604


In [131]:
# 1.4.5 Decision Tree Hyperparameter Tuning

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt_param_grid = {
    "max_depth": [3, 5, 8, 12, 16],
    "min_samples_split": [20, 50, 100],
    "min_samples_leaf": [10, 20, 50],
    "class_weight": [None, "balanced"]
}

dt_grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(
        random_state=42
    ),
    param_grid=dt_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    pre_dispatch="2*n_jobs",
    refit=True
)

dt_grid_search.fit(
    X_train_encoded,
    y_train_working
)

,estimator,DecisionTreeC...ndom_state=42)
,param_grid,"{'class_weight': [None, 'balanced'], 'max_depth': [3, 5, ...], 'min_samples_leaf': [10, 20, ...], 'min_samples_split': [20, 50, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'gini'


In [132]:
# Best Decision Tree parameters

best_dt_params = dt_grid_search.best_params_
best_dt_cv_f1 = dt_grid_search.best_score_

pd.DataFrame({
    "Best_Max_Depth": [
        best_dt_params["max_depth"]
    ],
    "Best_Min_Samples_Split": [
        best_dt_params["min_samples_split"]
    ],
    "Best_Min_Samples_Leaf": [
        best_dt_params["min_samples_leaf"]
    ],
    "Best_Class_Weight": [
        best_dt_params["class_weight"]
    ],
    "Best_CV_F1": [
        best_dt_cv_f1
    ]
})

,Best_Max_Depth,Best_Min_Samples_Split,Best_Min_Samples_Leaf,Best_Class_Weight,Best_CV_F1
0,3,20,10,balanced,0.252726


In [133]:
# Decision Tree tuning results

dt_tuning_results = pd.DataFrame(
    dt_grid_search.cv_results_
)

dt_tuning_results[
    [
        "param_max_depth",
        "param_min_samples_split",
        "param_min_samples_leaf",
        "param_class_weight",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
).reset_index(drop=True)

,param_max_depth,param_min_samples_split,param_min_samples_leaf,param_class_weight,mean_test_score,std_test_score,rank_test_score
0,3,50,50,balanced,0.252726,0.000860,1
1,3,20,10,balanced,0.252726,0.000860,1
2,3,50,10,balanced,0.252726,0.000860,1
3,3,100,10,balanced,0.252726,0.000860,1
4,3,20,20,balanced,0.252726,0.000860,1
...,...,...,...,...,...,...,...
85,3,50,50,None,0.004149,0.005867,85
86,3,20,50,None,0.004149,0.005867,85
87,3,100,20,None,0.004149,0.005867,85
88,3,50,20,None,0.004149,0.005867,85


In [134]:
# Tuned Decision Tree

tuned_dt_model = dt_grid_search.best_estimator_

tuned_dt_train_pred = tuned_dt_model.predict(
    X_train_encoded
)

tuned_dt_test_pred = tuned_dt_model.predict(
    X_test_encoded
)

tuned_dt_train_proba = tuned_dt_model.predict_proba(
    X_train_encoded
)[:, 1]

tuned_dt_test_proba = tuned_dt_model.predict_proba(
    X_test_encoded
)[:, 1]

tuned_dt_train_f1 = f1_score(
    y_train_working,
    tuned_dt_train_pred,
    zero_division=0
)

tuned_dt_test_f1 = f1_score(
    y_test_working,
    tuned_dt_test_pred,
    zero_division=0
)

pd.DataFrame({
    "Model": ["Tuned Decision Tree"],
    "Train_F1": [tuned_dt_train_f1],
    "Test_F1": [tuned_dt_test_f1],
    "F1_Gap": [
        tuned_dt_train_f1 - tuned_dt_test_f1
    ],
    "Best_CV_F1": [
        dt_grid_search.best_score_
    ]
})

,Model,Train_F1,Test_F1,F1_Gap,Best_CV_F1
0,Tuned Decision Tree,0.252714,0.255478,-0.002764,0.252726


## Next — 1.4.6 Random Forest Hyperparameter Tuning

- Random Forest was severely overfitting:

- Train F1 = 1.000000
- Test F1  = 0.019189
- CV F1    = 0.010075

In [135]:
# 1.4.6 Random Forest Hyperparameter Tuning

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf_param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [5, 8, 12],
    "min_samples_split": [20, 50],
    "min_samples_leaf": [10, 20],
    "max_features": ["sqrt"],
    "class_weight": ["balanced"]
}

rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=rf_param_grid,
    scoring="f1",
    cv=cv_strategy,
    n_jobs=-1,
    pre_dispatch="2*n_jobs",
    refit=True
)

rf_grid_search.fit(
    X_train_encoded,
    y_train_working
)

,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'class_weight': ['balanced'], 'max_depth': [5, 8, ...], 'max_features': ['sqrt'], 'min_samples_leaf': [10, 20], ...}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,300


In [136]:
# Best Random Forest parameters

best_rf_params = rf_grid_search.best_params_
best_rf_cv_f1 = rf_grid_search.best_score_

pd.DataFrame({
    "Best_n_estimators": [
        best_rf_params["n_estimators"]
    ],
    "Best_Max_Depth": [
        best_rf_params["max_depth"]
    ],
    "Best_Min_Samples_Split": [
        best_rf_params["min_samples_split"]
    ],
    "Best_Min_Samples_Leaf": [
        best_rf_params["min_samples_leaf"]
    ],
    "Best_Max_Features": [
        best_rf_params["max_features"]
    ],
    "Best_Class_Weight": [
        best_rf_params["class_weight"]
    ],
    "Best_CV_F1": [
        best_rf_cv_f1
    ]
})

,Best_n_estimators,Best_Max_Depth,Best_Min_Samples_Split,Best_Min_Samples_Leaf,Best_Max_Features,Best_Class_Weight,Best_CV_F1
0,300,12,20,20,sqrt,balanced,0.262335


In [137]:
'RF object exists:', 'rf_grid_search' in globals()

('RF object exists:', True)

In [138]:
# Random Forest tuning results

rf_tuning_results = pd.DataFrame(
    rf_grid_search.cv_results_
)

rf_tuning_results[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_min_samples_split",
        "param_min_samples_leaf",
        "param_max_features",
        "param_class_weight",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
).reset_index(drop=True)

,param_n_estimators,param_max_depth,param_min_samples_split,param_min_samples_leaf,param_max_features,param_class_weight,mean_test_score,std_test_score,rank_test_score
0,300,12,20,20,sqrt,balanced,0.262335,0.002688,1
1,200,12,20,20,sqrt,balanced,0.262044,0.002485,2
2,300,12,50,10,sqrt,balanced,0.260686,0.002147,3
3,300,12,50,20,sqrt,balanced,0.260653,0.001342,4
4,200,12,50,10,sqrt,balanced,0.260652,0.001637,5
5,200,12,50,20,sqrt,balanced,0.260130,0.001412,6
6,200,12,20,10,sqrt,balanced,0.259497,0.002094,7
7,200,8,50,20,sqrt,balanced,0.259386,0.003339,8
8,300,8,20,20,sqrt,balanced,0.259266,0.003156,9
9,300,12,20,10,sqrt,balanced,0.259199,0.003194,10


In [139]:
# Tuned Random Forest

tuned_rf_model = rf_grid_search.best_estimator_

tuned_rf_train_pred = tuned_rf_model.predict(
    X_train_encoded
)

tuned_rf_test_pred = tuned_rf_model.predict(
    X_test_encoded
)

tuned_rf_train_proba = tuned_rf_model.predict_proba(
    X_train_encoded
)[:, 1]

tuned_rf_test_proba = tuned_rf_model.predict_proba(
    X_test_encoded
)[:, 1]

tuned_rf_train_f1 = f1_score(
    y_train_working,
    tuned_rf_train_pred,
    zero_division=0
)

tuned_rf_test_f1 = f1_score(
    y_test_working,
    tuned_rf_test_pred,
    zero_division=0
)

pd.DataFrame({
    "Model": ["Tuned Random Forest"],
    "Train_F1": [tuned_rf_train_f1],
    "Test_F1": [tuned_rf_test_f1],
    "F1_Gap": [
        tuned_rf_train_f1 - tuned_rf_test_f1
    ],
    "Best_CV_F1": [
        rf_grid_search.best_score_
    ]
})

,Model,Train_F1,Test_F1,F1_Gap,Best_CV_F1
0,Tuned Random Forest,0.31652,0.265872,0.050648,0.262335


## 1.4.7 XGBoost Hyperparameter Tuning

- XGBoost baseline was very poor:

- Train F1 = 0.069955
- Test F1  = 0.027539
- CV F1    = 0.028026

In [140]:
# 1.4.7 XGBoost Hyperparameter Tuning

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

negative_count = (y_train_working == 0).sum()
positive_count = (y_train_working == 1).sum()

scale_pos_ratio = negative_count / positive_count

scale_pos_ratio

np.float64(7.780687216882113)

In [141]:
# XGBoost hyperparameter search space

xgb_param_distributions = {
    "n_estimators": [200, 300, 400],
    "max_depth": [2, 3, 5, 7],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 5, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "scale_pos_weight": [1.0, 5.0, scale_pos_ratio]
}

xgb_random_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=xgb_param_distributions,
    n_iter=20,
    scoring="f1",
    cv=cv_strategy,
    random_state=42,
    n_jobs=1,
    refit=True
)

xgb_random_search.fit(
    X_train_encoded,
    y_train_working
)

,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.7, 0.8, ...], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [2, 3, ...], 'min_child_weight': [1, 5, ...], ...}"
,n_iter,20
,scoring,'f1'
,n_jobs,1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [142]:
# Best XGBoost parameters

best_xgb_params = xgb_random_search.best_params_
best_xgb_cv_f1 = xgb_random_search.best_score_

pd.DataFrame({
    "Best_n_estimators": [
        best_xgb_params["n_estimators"]
    ],
    "Best_Max_Depth": [
        best_xgb_params["max_depth"]
    ],
    "Best_Learning_Rate": [
        best_xgb_params["learning_rate"]
    ],
    "Best_Min_Child_Weight": [
        best_xgb_params["min_child_weight"]
    ],
    "Best_Subsample": [
        best_xgb_params["subsample"]
    ],
    "Best_Colsample_Bytree": [
        best_xgb_params["colsample_bytree"]
    ],
    "Best_Scale_Pos_Weight": [
        best_xgb_params["scale_pos_weight"]
    ],
    "Best_CV_F1": [
        best_xgb_cv_f1
    ]
})

,Best_n_estimators,Best_Max_Depth,Best_Learning_Rate,Best_Min_Child_Weight,Best_Subsample,Best_Colsample_Bytree,Best_Scale_Pos_Weight,Best_CV_F1
0,400,2,0.1,5,1.0,1.0,7.780687,0.257871


In [143]:
# XGBoost tuning results

xgb_tuning_results = pd.DataFrame(
    xgb_random_search.cv_results_
)

xgb_tuning_results[
    [
        "param_n_estimators",
        "param_max_depth",
        "param_learning_rate",
        "param_min_child_weight",
        "param_subsample",
        "param_colsample_bytree",
        "param_scale_pos_weight",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
).reset_index(drop=True)

,param_n_estimators,param_max_depth,param_learning_rate,param_min_child_weight,param_subsample,param_colsample_bytree,param_scale_pos_weight,mean_test_score,std_test_score,rank_test_score
0,400,2,0.10,5,1.0,1.0,7.780687,0.257871,0.001366,1
1,300,3,0.10,5,1.0,1.0,7.780687,0.257287,0.001693,2
2,400,3,0.03,10,0.8,1.0,7.780687,0.257001,0.001947,3
3,400,3,0.05,5,1.0,0.7,7.780687,0.256903,0.000501,4
4,300,7,0.05,5,0.8,0.8,7.780687,0.256309,0.005699,5
5,400,7,0.03,10,1.0,1.0,7.780687,0.253145,0.003121,6
6,200,7,0.03,1,1.0,1.0,7.780687,0.252173,0.001643,7
7,200,7,0.10,1,0.8,0.7,7.780687,0.245905,0.004239,8
8,300,3,0.05,10,0.8,1.0,5.000000,0.244457,0.006821,9
9,300,2,0.10,5,0.8,0.8,5.000000,0.243763,0.006146,10


In [144]:
# Tuned XGBoost

tuned_xgb_model = xgb_random_search.best_estimator_

tuned_xgb_train_pred = tuned_xgb_model.predict(
    X_train_encoded
)

tuned_xgb_test_pred = tuned_xgb_model.predict(
    X_test_encoded
)

tuned_xgb_train_proba = tuned_xgb_model.predict_proba(
    X_train_encoded
)[:, 1]

tuned_xgb_test_proba = tuned_xgb_model.predict_proba(
    X_test_encoded
)[:, 1]

tuned_xgb_train_f1 = f1_score(
    y_train_working,
    tuned_xgb_train_pred,
    zero_division=0
)

tuned_xgb_test_f1 = f1_score(
    y_test_working,
    tuned_xgb_test_pred,
    zero_division=0
)

pd.DataFrame({
    "Model": ["Tuned XGBoost"],
    "Train_F1": [tuned_xgb_train_f1],
    "Test_F1": [tuned_xgb_test_f1],
    "F1_Gap": [
        tuned_xgb_train_f1 - tuned_xgb_test_f1
    ],
    "Best_CV_F1": [
        xgb_random_search.best_score_
    ]
})

,Model,Train_F1,Test_F1,F1_Gap,Best_CV_F1
0,Tuned XGBoost,0.273457,0.263141,0.010316,0.257871


In [145]:
# Create the best tuned Logistic Regression model

tuned_lr_model = lr_grid_search.best_estimator_

# Predictions
tuned_lr_train_pred = tuned_lr_model.predict(
    X_train_scaled
)

tuned_lr_test_pred = tuned_lr_model.predict(
    X_test_scaled
)

# Probabilities
tuned_lr_train_proba = tuned_lr_model.predict_proba(
    X_train_scaled
)[:, 1]

tuned_lr_test_proba = tuned_lr_model.predict_proba(
    X_test_scaled
)[:, 1]

# F1 scores
tuned_lr_train_f1 = f1_score(
    y_train_working,
    tuned_lr_train_pred,
    zero_division=0
)

tuned_lr_test_f1 = f1_score(
    y_test_working,
    tuned_lr_test_pred,
    zero_division=0
)

# Summary
pd.DataFrame({
    "Model": ["Tuned Logistic Regression"],
    "Train_F1": [tuned_lr_train_f1],
    "Test_F1": [tuned_lr_test_f1],
    "F1_Gap": [
        tuned_lr_train_f1 - tuned_lr_test_f1
    ],
    "Best_CV_F1": [
        lr_grid_search.best_score_
    ]
})

,Model,Train_F1,Test_F1,F1_Gap,Best_CV_F1
0,Tuned Logistic Regression,0.254314,0.253953,0.000361,0.2537


In [146]:
# Final tuned model comparison

tuned_model_results = []

tuned_models_data = {
    "Logistic Regression": (
        tuned_lr_train_pred,
        tuned_lr_test_pred,
        lr_grid_search.best_score_
    ),
    "KNN": (
        tuned_knn_train_pred,
        tuned_knn_test_pred,
        knn_grid_search.best_score_
    ),
    "Naive Bayes": (
        tuned_nb_train_pred,
        tuned_nb_test_pred,
        nb_grid_search.best_score_
    ),
    "SVM": (
        tuned_svm_train_pred,
        tuned_svm_test_pred,
        svm_grid_search.best_score_
    ),
    "Decision Tree": (
        tuned_dt_train_pred,
        tuned_dt_test_pred,
        dt_grid_search.best_score_
    ),
    "Random Forest": (
        tuned_rf_train_pred,
        tuned_rf_test_pred,
        rf_grid_search.best_score_
    ),
    "XGBoost": (
        tuned_xgb_train_pred,
        tuned_xgb_test_pred,
        xgb_random_search.best_score_
    )
}

for model_name, (
    train_pred,
    test_pred,
    cv_f1
) in tuned_models_data.items():

    train_f1 = f1_score(
        y_train_working,
        train_pred,
        zero_division=0
    )

    test_f1 = f1_score(
        y_test_working,
        test_pred,
        zero_division=0
    )

    f1_gap = train_f1 - test_f1

    # Same fit logic used for baseline
    if train_f1 < 0.50 and test_f1 < 0.50:
        fit = "Underfitting"
    elif f1_gap > 0.10:
        fit = "Overfitting"
    else:
        fit = "Good Fit"

    tuned_model_results.append({
        "Model": model_name,
        "Train_F1": train_f1,
        "Test_F1": test_f1,
        "F1_Gap": f1_gap,
        "Bias_Variance_Fit": fit,
        "CV_F1_Mean": cv_f1
    })

tuned_model_comparison = (
    pd.DataFrame(tuned_model_results)
    .sort_values("Test_F1", ascending=False)
    .reset_index(drop=True)
)

tuned_model_comparison

,Model,Train_F1,Test_F1,F1_Gap,Bias_Variance_Fit,CV_F1_Mean
0,Random Forest,0.316520,0.265872,0.050648,Underfitting,0.262335
1,XGBoost,0.273457,0.263141,0.010316,Underfitting,0.257871
2,Decision Tree,0.252714,0.255478,-0.002764,Underfitting,0.252726
3,SVM,0.253095,0.254509,-0.001414,Underfitting,0.252290
4,Logistic Regression,0.254314,0.253953,0.000361,Underfitting,0.253700
5,Naive Bayes,0.251239,0.252929,-0.001690,Underfitting,0.249893
6,KNN,0.966693,0.049645,0.917048,Overfitting,0.046430


In [147]:
def classify_fit_by_gap(f1_gap):
    gap = abs(f1_gap)

    if gap <= 0.05:
        return "Good Fit"
    elif gap <= 0.10:
        return "Moderate Variance"
    else:
        return "Overfitting"

tuned_model_comparison["Bias_Variance_Fit"] = (
    tuned_model_comparison["F1_Gap"]
    .apply(classify_fit_by_gap)
)

tuned_model_comparison

,Model,Train_F1,Test_F1,F1_Gap,Bias_Variance_Fit,CV_F1_Mean
0,Random Forest,0.316520,0.265872,0.050648,Moderate Variance,0.262335
1,XGBoost,0.273457,0.263141,0.010316,Good Fit,0.257871
2,Decision Tree,0.252714,0.255478,-0.002764,Good Fit,0.252726
3,SVM,0.253095,0.254509,-0.001414,Good Fit,0.252290
4,Logistic Regression,0.254314,0.253953,0.000361,Good Fit,0.253700
5,Naive Bayes,0.251239,0.252929,-0.001690,Good Fit,0.249893
6,KNN,0.966693,0.049645,0.917048,Overfitting,0.046430


## 1.5 Tuned Model Detailed Evaluation

First, calibrate the tuned SVM so we can obtain probabilities and a proper Brier score.

# 1.5.1 Calibrate Tuned SVM

In [148]:
# 1.5.1 Calibrate Tuned SVM

from sklearn.calibration import CalibratedClassifierCV

calibrated_tuned_svm = CalibratedClassifierCV(
    estimator=tuned_svm_model,
    method="sigmoid",
    cv=3,
    n_jobs=-1
)

calibrated_tuned_svm.fit(
    X_train_scaled,
    y_train_working
)

tuned_svm_calibrated_proba = (
    calibrated_tuned_svm.predict_proba(
        X_test_scaled
    )[:, 1]
)

In [149]:
## 1.5.2 Evaluate all tuned models
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

tuned_predictions = {
    "Logistic Regression": tuned_lr_test_pred,
    "KNN": tuned_knn_test_pred,
    "Naive Bayes": tuned_nb_test_pred,
    "SVM": tuned_svm_test_pred,
    "Decision Tree": tuned_dt_test_pred,
    "Random Forest": tuned_rf_test_pred,
    "XGBoost": tuned_xgb_test_pred
}

tuned_scores = {
    "Logistic Regression": tuned_lr_test_proba,
    "KNN": tuned_knn_test_proba,
    "Naive Bayes": tuned_nb_test_proba,
    "SVM": tuned_svm_calibrated_proba,
    "Decision Tree": tuned_dt_test_proba,
    "Random Forest": tuned_rf_test_proba,
    "XGBoost": tuned_xgb_test_proba
}

tuned_evaluation_results = []

for model_name in tuned_predictions:

    predictions = tuned_predictions[model_name]
    probabilities = tuned_scores[model_name]

    tuned_evaluation_results.append({
        "Model": model_name,
        "Precision": precision_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test_working,
            probabilities
        ),
        "PR_AUC": average_precision_score(
            y_test_working,
            probabilities
        ),
        "Brier_Score": brier_score_loss(
            y_test_working,
            probabilities
        )
    })

tuned_evaluation_results = (
    pd.DataFrame(tuned_evaluation_results)
    .sort_values("PR_AUC", ascending=False)
    .reset_index(drop=True)
)

tuned_evaluation_results

,Model,Precision,Recall,F1,ROC_AUC,PR_AUC,Brier_Score
0,Random Forest,0.188084,0.453380,0.265872,0.647273,0.211978,0.218880
1,XGBoost,0.172970,0.549713,0.263141,0.650544,0.210509,0.227634
2,Logistic Regression,0.175712,0.457799,0.253953,0.636177,0.203064,0.233318
3,SVM,0.177112,0.452055,0.254509,0.634220,0.201004,0.097805
4,Naive Bayes,0.181906,0.414936,0.252929,0.631174,0.195782,0.229279
5,Decision Tree,0.171598,0.499779,0.255478,0.627274,0.180606,0.233842
6,KNN,0.229091,0.027839,0.049645,0.576073,0.146009,0.112280


## Next — Tuned Confusion Matrix



In [150]:
# 1.5.2 Tuned Model Confusion Matrices

tuned_test_predictions = {
    "Logistic Regression": tuned_lr_test_pred,
    "KNN": tuned_knn_test_pred,
    "Naive Bayes": tuned_nb_test_pred,
    "SVM": tuned_svm_test_pred,
    "Decision Tree": tuned_dt_test_pred,
    "Random Forest": tuned_rf_test_pred,
    "XGBoost": tuned_xgb_test_pred
}

tuned_confusion_matrices = {}

for model_name, predictions in tuned_test_predictions.items():

    tuned_confusion_matrices[model_name] = confusion_matrix(
        y_test_working,
        predictions
    )

for model_name, cm in tuned_confusion_matrices.items():

    tn, fp, fn, tp = cm.ravel()

    print(f"\n{model_name}")
    print(f"TN = {tn}")
    print(f"FP = {fp}")
    print(f"FN = {fn}")
    print(f"TP = {tp}")


Logistic Regression
TN = 12746
FP = 4860
FN = 1227
TP = 1036

KNN
TN = 17394
FP = 212
FN = 2200
TP = 63

Naive Bayes
TN = 13383
FP = 4223
FN = 1324
TP = 939

SVM
TN = 12853
FP = 4753
FN = 1240
TP = 1023

Decision Tree
TN = 12146
FP = 5460
FN = 1132
TP = 1131

Random Forest
TN = 13177
FP = 4429
FN = 1237
TP = 1026

XGBoost
TN = 11658
FP = 5948
FN = 1019
TP = 1244


In [151]:
tuned_confusion_summary = []

for model_name, cm in tuned_confusion_matrices.items():

    tn, fp, fn, tp = cm.ravel()

    tuned_confusion_summary.append({
        "Model": model_name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

tuned_confusion_summary = pd.DataFrame(
    tuned_confusion_summary
)

tuned_confusion_summary

,Model,TN,FP,FN,TP
0,Logistic Regression,12746,4860,1227,1036
1,KNN,17394,212,2200,63
2,Naive Bayes,13383,4223,1324,939
3,SVM,12853,4753,1240,1023
4,Decision Tree,12146,5460,1132,1131
5,Random Forest,13177,4429,1237,1026
6,XGBoost,11658,5948,1019,1244


##  Threshold Optimization.

- We should do this correctly: do not use the final test set to choose the threshold. We'll split the training data into an inner-training set and validation set, fit the four strongest candidates again using their already-tuned hyperparameters, and use only the validation set to find the best threshold.

- We'll start with:

- Random Forest, XGBoost, Logistic Regression, SVM

## 1.6.1 Create an inner validation set

In [152]:
# 1.6.1 Threshold Optimization
# Create an inner validation split from training data

from sklearn.model_selection import train_test_split

X_rf_subtrain, X_rf_valid, y_subtrain, y_valid = train_test_split(
    X_train_encoded,
    y_train_working,
    test_size=0.20,
    random_state=42,
    stratify=y_train_working
)

X_scaled_subtrain, X_scaled_valid, _, _ = train_test_split(
    X_train_scaled,
    y_train_working,
    test_size=0.20,
    random_state=42,
    stratify=y_train_working
)

X_rf_subtrain.shape, X_rf_valid.shape, y_subtrain.shape, y_valid.shape

((63579, 215), (15895, 215), (63579,), (15895,))

## 1.6.2 Refit tuned candidate models on inner-training data

In [153]:
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV

# Clone the already-tuned models
threshold_lr_model = clone(tuned_lr_model)
threshold_rf_model = clone(tuned_rf_model)
threshold_xgb_model = clone(tuned_xgb_model)
threshold_svm_base = clone(tuned_svm_model)

# Train Logistic Regression
threshold_lr_model.fit(
    X_scaled_subtrain,
    y_subtrain
)

# Train Random Forest
threshold_rf_model.fit(
    X_rf_subtrain,
    y_subtrain
)

# Train XGBoost
threshold_xgb_model.fit(
    X_rf_subtrain,
    y_subtrain
)

# Calibrate SVM using only the inner-training data
threshold_svm_model = CalibratedClassifierCV(
    estimator=threshold_svm_base,
    method="sigmoid",
    cv=3,
    n_jobs=-1
)

threshold_svm_model.fit(
    X_scaled_subtrain,
    y_subtrain
)

,estimator,LinearSVC(C=0...ndom_state=42)
,method,'sigmoid'
,cv,3
,n_jobs,-1
,ensemble,'auto'
,penalty,'l2'
,loss,'squared_hinge'
,dual,False
,tol,0.0001
,C,0.01
,multi_class,'ovr'


## 1.6.3 Get validation probabilities

In [154]:
# Validation probabilities

validation_probabilities = {
    "Logistic Regression": threshold_lr_model.predict_proba(
        X_scaled_valid
    )[:, 1],

    "Random Forest": threshold_rf_model.predict_proba(
        X_rf_valid
    )[:, 1],

    "XGBoost": threshold_xgb_model.predict_proba(
        X_rf_valid
    )[:, 1],

    "SVM": threshold_svm_model.predict_proba(
        X_scaled_valid
    )[:, 1]
}

## 1.6.4 Find the best threshold by validation F1

In [155]:
# Threshold search

threshold_results = []

threshold_values = np.arange(
    0.10,
    0.91,
    0.01
)

for model_name, probabilities in validation_probabilities.items():

    for threshold in threshold_values:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_valid,
            predictions,
            zero_division=0
        )

        recall = recall_score(
            y_valid,
            predictions,
            zero_division=0
        )

        f1 = f1_score(
            y_valid,
            predictions,
            zero_division=0
        )

        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        })

threshold_results = pd.DataFrame(
    threshold_results
)

best_threshold_results = (
    threshold_results
    .sort_values(
        ["Model", "F1"],
        ascending=[True, False]
    )
    .groupby("Model")
    .head(1)
    .reset_index(drop=True)
)

best_threshold_results

,Model,Threshold,Precision,Recall,F1
0,Logistic Regression,0.52,0.187647,0.397790,0.255003
1,Random Forest,0.50,0.182636,0.448619,0.259591
2,SVM,0.12,0.182259,0.415470,0.253369
3,XGBoost,0.51,0.171979,0.512707,0.257563


## Final Test Evaluation at Optimized Thresholds

In [156]:
# Final test evaluation using validation-selected thresholds

selected_thresholds = {
    "Logistic Regression": 0.53,
    "Random Forest": 0.50,
    "SVM": 0.13,
    "XGBoost": 0.51
}

final_threshold_results = []

final_threshold_probabilities = {
    "Logistic Regression": tuned_lr_test_proba,
    "Random Forest": tuned_rf_test_proba,
    "SVM": tuned_svm_calibrated_proba,
    "XGBoost": tuned_xgb_test_proba
}

for model_name, threshold in selected_thresholds.items():

    probabilities = final_threshold_probabilities[model_name]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    final_threshold_results.append({
        "Model": model_name,
        "Threshold": threshold,
        "Precision": precision_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test_working,
            probabilities
        ),
        "PR_AUC": average_precision_score(
            y_test_working,
            probabilities
        ),
        "Brier_Score": brier_score_loss(
            y_test_working,
            probabilities
        )
    })

final_threshold_results = (
    pd.DataFrame(final_threshold_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

final_threshold_results

,Model,Threshold,Precision,Recall,F1,ROC_AUC,PR_AUC,Brier_Score
0,Random Forest,0.50,0.188084,0.453380,0.265872,0.647273,0.211978,0.218880
1,XGBoost,0.51,0.177690,0.518780,0.264713,0.650544,0.210509,0.227634
2,SVM,0.13,0.197663,0.351304,0.252983,0.634220,0.201004,0.097805
3,Logistic Regression,0.53,0.192567,0.366328,0.252436,0.636177,0.203064,0.233318


In [157]:
# Final test confusion matrices at optimized thresholds

final_threshold_confusion = []

for model_name, threshold in selected_thresholds.items():

    probabilities = final_threshold_probabilities[model_name]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_working,
        predictions
    ).ravel()

    final_threshold_confusion.append({
        "Model": model_name,
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

final_threshold_confusion = pd.DataFrame(
    final_threshold_confusion
)

final_threshold_confusion

,Model,Threshold,TN,FP,FN,TP
0,Logistic Regression,0.53,14130,3476,1434,829
1,Random Forest,0.50,13177,4429,1237,1026
2,SVM,0.13,14379,3227,1468,795
3,XGBoost,0.51,12173,5433,1089,1174


## Random Forest is the strongest balanced model because it has the highest final F1 and PR-AUC.

- XGBoost is the stronger model for the actual care-management objective because it catches more actual readmissions:

- XGBoost → Recall = 51.88%
- Random Forest → Recall = 45.34%

- XGBoost also has the highest ROC-AUC.

So we have two serious final candidates:

- Random Forest → best F1 / PR-AUC
- XGBoost       → best Recall / ROC-AUC

## One important observation

The validation-selected thresholds did not generalize equally well:

- RF: threshold stayed at 0.50, so performance stayed the same.
- XGBoost: 0.51 gave a small F1 improvement.
- LR and SVM: their validation-selected thresholds actually reduced final test F1.

## Calibration

Because this is a risk prediction system, not merely a classifier, we should now check whether the predicted probabilities represent realistic risk.

The Brier results show a major calibration issue for the tree models, while calibrated SVM is much better. So before selecting the final model, we should calibrate the two strongest candidates:

- Random Forest + XGBoost

- using training data only, then evaluate:

- Brier Score + calibration curve + final Recall/F1/PR-AUC.

## 1.6 Calibration — Random Forest & XGBoost
## Calibrate both models

In [158]:
# 1.6.1 Probability Calibration
# Calibration is fitted only on the inner training data.
# The final test set remains completely untouched.

from sklearn.calibration import CalibratedClassifierCV

calibrated_rf = CalibratedClassifierCV(
    estimator=tuned_rf_model,
    method="sigmoid",
    cv=3,
    n_jobs=-1
)

calibrated_xgb = CalibratedClassifierCV(
    estimator=tuned_xgb_model,
    method="sigmoid",
    cv=3,
    n_jobs=-1
)

calibrated_rf.fit(
    X_rf_subtrain,
    y_subtrain
)

calibrated_xgb.fit(
    X_rf_subtrain,
    y_subtrain
)

,estimator,"XGBClassifier...ree=None, ...)"
,method,'sigmoid'
,cv,3
,n_jobs,-1
,ensemble,'auto'
,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None


In [159]:
# Calibrated probabilities on the inner validation set

rf_valid_calibrated_proba = calibrated_rf.predict_proba(
    X_rf_valid
)[:, 1]

xgb_valid_calibrated_proba = calibrated_xgb.predict_proba(
    X_rf_valid
)[:, 1]

In [160]:
# Threshold optimization after calibration

calibrated_validation_scores = {
    "Random Forest": rf_valid_calibrated_proba,
    "XGBoost": xgb_valid_calibrated_proba
}

calibrated_threshold_results = []

threshold_values = np.arange(
    0.10,
    0.91,
    0.01
)

for model_name, probabilities in calibrated_validation_scores.items():

    for threshold in threshold_values:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        calibrated_threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Precision": precision_score(
                y_valid,
                predictions,
                zero_division=0
            ),
            "Recall": recall_score(
                y_valid,
                predictions,
                zero_division=0
            ),
            "F1": f1_score(
                y_valid,
                predictions,
                zero_division=0
            )
        })

calibrated_threshold_results = pd.DataFrame(
    calibrated_threshold_results
)

best_calibrated_thresholds = (
    calibrated_threshold_results
    .sort_values(
        ["Model", "F1"],
        ascending=[True, False]
    )
    .groupby("Model")
    .head(1)
    .reset_index(drop=True)
)

best_calibrated_thresholds

,Model,Threshold,Precision,Recall,F1
0,Random Forest,0.15,0.208448,0.335359,0.257094
1,XGBoost,0.12,0.171298,0.517680,0.257418


In [161]:
# Final calibrated probabilities on test data

rf_test_calibrated_proba = calibrated_rf.predict_proba(
    X_test_encoded
)[:, 1]

xgb_test_calibrated_proba = calibrated_xgb.predict_proba(
    X_test_encoded
)[:, 1]

In [162]:
# Final test evaluation after calibration and validation-selected threshold

calibrated_test_models = {
    "Random Forest": rf_test_calibrated_proba,
    "XGBoost": xgb_test_calibrated_proba
}

calibrated_final_results = []

for model_name, probabilities in calibrated_test_models.items():

    threshold = best_calibrated_thresholds.loc[
        best_calibrated_thresholds["Model"] == model_name,
        "Threshold"
    ].iloc[0]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    calibrated_final_results.append({
        "Model": model_name,
        "Threshold": threshold,
        "Precision": precision_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test_working,
            predictions,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test_working,
            probabilities
        ),
        "PR_AUC": average_precision_score(
            y_test_working,
            probabilities
        ),
        "Brier_Score": brier_score_loss(
            y_test_working,
            probabilities
        )
    })

calibrated_final_results = (
    pd.DataFrame(calibrated_final_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

calibrated_final_results

,Model,Threshold,Precision,Recall,F1,ROC_AUC,PR_AUC,Brier_Score
0,XGBoost,0.12,0.178465,0.530270,0.267052,0.649888,0.209674,0.097364
1,Random Forest,0.15,0.205407,0.325674,0.251923,0.647212,0.211444,0.097476


In [163]:
# Compare calibrated probability quality

calibration_summary = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "Brier_Score": [
        brier_score_loss(
            y_test_working,
            rf_test_calibrated_proba
        ),
        brier_score_loss(
            y_test_working,
            xgb_test_calibrated_proba
        )
    ]
})

calibration_summary

,Model,Brier_Score
0,Random Forest,0.097476
1,XGBoost,0.097364


In [164]:
# Final XGBoost confusion matrix after calibration

final_xgb_threshold = 0.12

final_xgb_pred = (
    xgb_test_calibrated_proba >= final_xgb_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test_working,
    final_xgb_pred
).ravel()

final_xgb_confusion = pd.DataFrame({
    "Metric": ["TN", "FP", "FN", "TP"],
    "Count": [tn, fp, fn, tp]
})

final_xgb_confusion

,Metric,Count
0,TN,12082
1,FP,5524
2,FN,1063
3,TP,1200


## ### Validation — Can Exact Days to Readmission Be Derived?

In [165]:
import pandas as pd
from IPython.display import display

dataframes_found = []

# Make a fixed snapshot before iterating
namespace_items = list(globals().items())

for variable_name, variable_value in namespace_items:
    if isinstance(variable_value, pd.DataFrame):
        dataframes_found.append({
            "variable": variable_name,
            "rows": variable_value.shape[0],
            "columns": variable_value.shape[1],
            "has_readmitted": "readmitted" in variable_value.columns,
            "has_patient_nbr": "patient_nbr" in variable_value.columns,
            "has_encounter_id": "encounter_id" in variable_value.columns
        })

dataframes_found = pd.DataFrame(dataframes_found)

display(dataframes_found)

,variable,rows,columns,has_readmitted,has_patient_nbr,has_encounter_id
0,__,7,7,False,False,False
1,___,7,6,False,False,False
2,df_model_classification,101766,50,True,True,True
3,ids_mapping_classification,67,2,False,False,False
4,admission_type_mapping,8,2,False,False,False
...,...,...,...,...,...,...
138,_162,2,8,False,False,False
139,calibration_summary,2,2,False,False,False
140,_163,2,2,False,False,False
141,final_xgb_confusion,4,2,False,False,False


In [166]:
df_model_classification

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide_metformin,glipizide_metformin,glimepiride_pioglitazone,metformin_rosiglitazone,metformin_pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101761,443847548,100162476,AfricanAmerican,Male,[70-80),?,1,3,7,3,...,No,Down,No,No,No,No,No,Ch,Yes,>30
101762,443847782,74694222,AfricanAmerican,Female,[80-90),?,1,4,5,5,...,No,Steady,No,No,No,No,No,No,Yes,NO
101763,443854148,41088789,Caucasian,Male,[70-80),?,1,1,7,1,...,No,Down,No,No,No,No,No,Ch,Yes,NO
101764,443857166,31693671,Caucasian,Female,[80-90),?,2,3,7,10,...,No,Up,No,No,No,No,No,Ch,Yes,NO


In [167]:
import pandas as pd
from IPython.display import display

# Use the verified original raw dataset already loaded in the notebook.
raw_df = df_model_classification.copy()

# ------------------------------------------------------------
# 1. Inspect columns related to time, admission, discharge,
#    dates, and readmission
# ------------------------------------------------------------
keyword_columns = [
    col for col in raw_df.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "date",
            "time",
            "admit",
            "admission",
            "discharge",
            "readmission"
        ]
    )
]

column_check = pd.DataFrame({
    "column": keyword_columns,
    "dtype": [raw_df[col].dtype for col in keyword_columns],
    "non_null_count": [
        raw_df[col].notna().sum()
        for col in keyword_columns
    ],
    "unique_values": [
        raw_df[col].nunique(dropna=True)
        for col in keyword_columns
    ]
})

display(column_check)


# ------------------------------------------------------------
# 2. Check for actual datetime columns
# ------------------------------------------------------------
datetime_columns = [
    col
    for col in raw_df.columns
    if pd.api.types.is_datetime64_any_dtype(raw_df[col])
]

datetime_check = pd.DataFrame({
    "datetime_column": datetime_columns
})

display(datetime_check)


# ------------------------------------------------------------
# 3. Verify the original readmission outcome
# ------------------------------------------------------------
readmission_check = (
    raw_df["readmitted"]
    .value_counts(dropna=False)
    .rename_axis("readmitted")
    .reset_index(name="count")
)

readmission_check["percentage"] = (
    readmission_check["count"] / len(raw_df) * 100
)

display(readmission_check)


# ------------------------------------------------------------
# 4. Check whether patients have multiple encounters
# ------------------------------------------------------------
patient_counts = raw_df["patient_nbr"].value_counts()

patient_check = pd.DataFrame({
    "total_rows": [len(raw_df)],
    "unique_patients": [raw_df["patient_nbr"].nunique()],
    "patients_with_multiple_encounters": [
        (patient_counts > 1).sum()
    ]
})

display(patient_check)


# ------------------------------------------------------------
# 5. Verify identifier fields
# ------------------------------------------------------------
identifier_check = pd.DataFrame({
    "field": [
        "encounter_id",
        "patient_nbr"
    ],
    "dtype": [
        str(raw_df["encounter_id"].dtype),
        str(raw_df["patient_nbr"].dtype)
    ],
    "unique_count": [
        raw_df["encounter_id"].nunique(),
        raw_df["patient_nbr"].nunique()
    ],
    "missing_count": [
        raw_df["encounter_id"].isna().sum(),
        raw_df["patient_nbr"].isna().sum()
    ]
})

display(identifier_check)


# ------------------------------------------------------------
# 6. Check whether exact readmission-day information exists
# ------------------------------------------------------------
has_readmission_date = any(
    "readmission" in col.lower() and "date" in col.lower()
    for col in raw_df.columns
)

has_admission_date = any(
    "admission" in col.lower() and "date" in col.lower()
    for col in raw_df.columns
)

has_discharge_date = any(
    "discharge" in col.lower() and "date" in col.lower()
    for col in raw_df.columns
)

exact_days_available = (
    has_readmission_date
    and has_admission_date
    and has_discharge_date
)

conclusion = pd.DataFrame({
    "validation_item": [
        "Readmission date available",
        "Admission date available",
        "Discharge date available",
        "Exact days to readmission derivable"
    ],
    "result": [
        has_readmission_date,
        has_admission_date,
        has_discharge_date,
        exact_days_available
    ]
})

display(conclusion)

,column,dtype,non_null_count,unique_values
0,admission_type_id,int64,101766,8
1,discharge_disposition_id,int64,101766,26
2,admission_source_id,int64,101766,17
3,time_in_hospital,int64,101766,14
4,readmitted,object,101766,3


,datetime_column


,readmitted,count,percentage
0,NO,54864,53.911916
1,>30,35545,34.928169
2,<30,11357,11.159916


,total_rows,unique_patients,patients_with_multiple_encounters
0,101766,71518,16773


,field,dtype,unique_count,missing_count
0,encounter_id,int64,101766,0
1,patient_nbr,int64,71518,0


,validation_item,result
0,Readmission date available,False
1,Admission date available,False
2,Discharge date available,False
3,Exact days to readmission derivable,False


In [168]:
# Find scaler / encoder / preprocessing objects currently loaded
preprocessing_objects = []

for name in list(globals().keys()):
    name_lower = name.lower()

    if any(
        keyword in name_lower
        for keyword in ["scaler", "encoder", "preprocessor"]
    ):
        obj = globals()[name]

        preprocessing_objects.append({
            "variable": name,
            "type": type(obj).__name__
        })

preprocessing_objects = pd.DataFrame(preprocessing_objects)

display(preprocessing_objects)

,variable,type
0,OrdinalEncoder,type
1,fs_encoder,OrdinalEncoder
2,OneHotEncoder,type
3,encoder,OneHotEncoder
4,StandardScaler,type
5,numeric_scaler,StandardScaler
6,saved_encoder,OneHotEncoder
7,saved_scaler,StandardScaler


In [169]:
# ============================================================
# FINAL 30-DAY HOSPITAL READMISSION MODEL
# Save exactly 3 PKL deployment artifacts
# ============================================================

from pathlib import Path
import pickle
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)


# ============================================================
# 1. Deployment directory
# ============================================================

deployment_dir = (
    Path(r"C:\Users\chinn\OneDrive\Desktop\AI-Hospital-Readmission")
    / "ml-fastapi"
    / "pickles"
    / "model2"
)

deployment_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. Use the EXACT fitted objects already created
# ============================================================

final_encoder = encoder
final_scaler = numeric_scaler
final_model = calibrated_xgb

FINAL_THRESHOLD = 0.12


# ============================================================
# 3. Verify final modeling matrices
# ============================================================

assert X_train_encoded.shape == (79474, 215), (
    f"Unexpected X_train_encoded shape: {X_train_encoded.shape}"
)

assert X_test_encoded.shape == (19869, 215), (
    f"Unexpected X_test_encoded shape: {X_test_encoded.shape}"
)

assert type(final_encoder).__name__ == "OneHotEncoder", (
    f"Unexpected encoder type: {type(final_encoder).__name__}"
)

assert type(final_scaler).__name__ == "StandardScaler", (
    f"Unexpected scaler type: {type(final_scaler).__name__}"
)


# ============================================================
# 4. Generate final test probabilities
# ============================================================

final_test_proba = final_model.predict_proba(
    X_test_encoded
)[:, 1]

assert len(final_test_proba) == len(y_test_working), (
    "Prediction length does not match test target length."
)


# ============================================================
# 5. Apply final decision threshold
# ============================================================

final_test_pred = (
    final_test_proba >= FINAL_THRESHOLD
).astype(int)


# ============================================================
# 6. Calculate metrics from the exact saved model
# ============================================================

final_metrics = {
    "precision": precision_score(
        y_test_working,
        final_test_pred,
        zero_division=0
    ),
    "recall": recall_score(
        y_test_working,
        final_test_pred,
        zero_division=0
    ),
    "f1": f1_score(
        y_test_working,
        final_test_pred,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test_working,
        final_test_proba
    ),
    "pr_auc": average_precision_score(
        y_test_working,
        final_test_proba
    ),
    "brier_score": brier_score_loss(
        y_test_working,
        final_test_proba
    ),
    "threshold": FINAL_THRESHOLD
}


# ============================================================
# 7. Frozen feature metadata
# ============================================================

selected_features = [
    "age",
    "gender",
    "race",
    "admission_type_id",
    "admission_source_id",
    "time_in_hospital",
    "medical_specialty",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "max_glu_serum",
    "A1Cresult",
    "num_medications",
    "num_lab_procedures",
    "num_procedures",
    "insulin",
    "diabetesMed",
    "change",
    "prior_utilization",
    "high_inpatient_utilization",
    "high_emergency_utilization",
    "diagnosis_complexity",
    "medication_burden_group",
    "age_group",
    "glucose_abnormal_flag",
    "insulin_change_flag",
    "diag_1_category",
    "diag_2_category",
    "diag_3_category",
    "multiple_prior_encounters_flag"
]

categorical_features = [
    "age",
    "gender",
    "race",
    "admission_type_id",
    "admission_source_id",
    "medical_specialty",
    "max_glu_serum",
    "A1Cresult",
    "insulin",
    "diabetesMed",
    "change",
    "diagnosis_complexity",
    "medication_burden_group",
    "age_group",
    "diag_1_category",
    "diag_2_category",
    "diag_3_category"
]

numerical_features = [
    "time_in_hospital",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "num_medications",
    "num_lab_procedures",
    "num_procedures",
    "prior_utilization",
    "high_inpatient_utilization",
    "high_emergency_utilization",
    "glucose_abnormal_flag",
    "insulin_change_flag",
    "multiple_prior_encounters_flag"
]

engineered_features = [
    "prior_utilization",
    "high_inpatient_utilization",
    "high_emergency_utilization",
    "diagnosis_complexity",
    "medication_burden_group",
    "age_group",
    "glucose_abnormal_flag",
    "insulin_change_flag",
    "diag_1_category",
    "diag_2_category",
    "diag_3_category",
    "multiple_prior_encounters_flag",
    "high_diagnosis_burden_flag"
]

frontend_input_features = [
    "age",
    "gender",
    "race",
    "admission_type_id",
    "admission_source_id",
    "time_in_hospital",
    "medical_specialty",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "diag_1",
    "diag_2",
    "diag_3",
    "max_glu_serum",
    "A1Cresult",
    "num_medications",
    "num_lab_procedures",
    "num_procedures",
    "insulin",
    "diabetesMed",
    "change"
]


# ============================================================
# 8. Create final deployment package
# ============================================================

model_package = {
    "model_name": "XGBoost",
    "model_type": "CalibratedClassifierCV",
    "task": "30-Day Hospital Readmission Risk Classification",

    "target_name": "target_readmit_30d",

    "target_definition": {
        "positive_class": "<30",
        "positive_mapping": 1,
        "negative_mapping": 0
    },

    "decision_threshold": FINAL_THRESHOLD,

    "frontend_input_features": frontend_input_features,
    "selected_features": selected_features,
    "categorical_features": categorical_features,
    "numerical_features": numerical_features,
    "engineered_features": engineered_features,

    "encoder": final_encoder,
    "scaler": final_scaler,
    "model": final_model,

    "metrics": final_metrics,

    "feature_dimensions": {
        "onehot_output": 201,
        "numerical_input": 14,
        "final_model_input": 215
    }
}


# ============================================================
# 9. File paths
# ============================================================

encoder_path = (
    deployment_dir / "classification_onehot_encoder.pkl"
)

scaler_path = (
    deployment_dir / "classification_scaler.pkl"
)

model_path = (
    deployment_dir / "final_xgboost_classifier.pkl"
)


# ============================================================
# 10. Save all three PKL files
# ============================================================

with open(encoder_path, "wb") as file:
    pickle.dump(
        final_encoder,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )

with open(scaler_path, "wb") as file:
    pickle.dump(
        final_scaler,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )

with open(model_path, "wb") as file:
    pickle.dump(
        model_package,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# ============================================================
# 11. Verify files were created
# ============================================================

artifact_verification = pd.DataFrame({
    "artifact": [
        encoder_path.name,
        scaler_path.name,
        model_path.name
    ],
    "exists": [
        encoder_path.is_file(),
        scaler_path.is_file(),
        model_path.is_file()
    ],
    "size_kb": [
        round(encoder_path.stat().st_size / 1024, 2),
        round(scaler_path.stat().st_size / 1024, 2),
        round(model_path.stat().st_size / 1024, 2)
    ]
})

display(artifact_verification)


# ============================================================
# 12. Reload all three files
# ============================================================

with open(encoder_path, "rb") as file:
    loaded_encoder = pickle.load(file)

with open(scaler_path, "rb") as file:
    loaded_scaler = pickle.load(file)

with open(model_path, "rb") as file:
    loaded_package = pickle.load(file)


# ============================================================
# 13. Reload verification
# ============================================================

assert type(loaded_encoder).__name__ == "OneHotEncoder"
assert type(loaded_scaler).__name__ == "StandardScaler"
assert "model" in loaded_package
assert loaded_package["model_name"] == "XGBoost"
assert loaded_package["decision_threshold"] == 0.12

loaded_test_proba = (
    loaded_package["model"]
    .predict_proba(X_test_encoded)[:, 1]
)

assert len(loaded_test_proba) == len(final_test_proba)

assert (
    loaded_test_proba == final_test_proba
).all(), (
    "Reloaded model probabilities do not match "
    "the original final model."
)


# ============================================================
# 14. Final results
# ============================================================

verification_summary = pd.DataFrame({
    "check": [
        "X_train_encoded features",
        "X_test_encoded features",
        "Encoder type",
        "Scaler type",
        "Threshold",
        "Encoder saved",
        "Scaler saved",
        "XGBoost saved",
        "Reloaded predictions identical"
    ],
    "result": [
        X_train_encoded.shape[1],
        X_test_encoded.shape[1],
        type(final_encoder).__name__,
        type(final_scaler).__name__,
        FINAL_THRESHOLD,
        encoder_path.is_file(),
        scaler_path.is_file(),
        model_path.is_file(),
        True
    ]
})

display(verification_summary)
display(pd.DataFrame([final_metrics]))

,artifact,exists,size_kb
0,classification_onehot_encoder.pkl,True,3.72
1,classification_scaler.pkl,True,0.76
2,final_xgboost_classifier.pkl,True,1438.41


,check,result
0,X_train_encoded features,215
1,X_test_encoded features,215
2,Encoder type,OneHotEncoder
3,Scaler type,StandardScaler
4,Threshold,0.12
5,Encoder saved,True
6,Scaler saved,True
7,XGBoost saved,True
8,Reloaded predictions identical,True


,precision,recall,f1,roc_auc,pr_auc,brier_score,threshold
0,0.178465,0.53027,0.267052,0.649888,0.209674,0.097364,0.12


In [170]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

# Recover the fitted classification encoder
classification_encoder_final = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

classification_encoder_final.fit(
    X_train_selected[categorical_features]
)

# Verification
classification_ohe_output = classification_encoder_final.transform(
    X_train_selected[categorical_features]
)

print("Classification encoder recovered successfully.")
print("Input features:", classification_encoder_final.n_features_in_)
print("Output features:", classification_ohe_output.shape[1])

Classification encoder recovered successfully.
Input features: 17
Output features: 201


In [173]:
assert classification_encoder_final.n_features_in_ == 17
assert recovered_train_encoded.shape[1] == 201

In [172]:
from pathlib import Path
import pickle
import shutil
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder


# ============================================================
# 1. Rebuild the correct fitted classification encoder
# ============================================================

classification_encoder_final = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)

classification_train_categorical = X_train_selected[
    categorical_features
].copy()

classification_encoder_final.fit(
    classification_train_categorical
)


# ============================================================
# 2. Transform training categorical features
# ============================================================

recovered_train_encoded = (
    classification_encoder_final
    .transform(classification_train_categorical)
)


# ============================================================
# 3. Verify encoder structure
# ============================================================

assert classification_encoder_final.n_features_in_ == 17

assert recovered_train_encoded.shape[1] == 201


# ============================================================
# 4. Optional consistency check against original X_train_cat
# ============================================================

if "X_train_cat" in globals():

    assert recovered_train_encoded.shape == X_train_cat.shape

    difference = (
        recovered_train_encoded != X_train_cat
    ).nnz

    assert difference == 0, (
        f"Recovered encoder differs from the original "
        f"X_train_cat in {difference} positions."
    )


# ============================================================
# 5. Deployment paths
# ============================================================

deployment_dir = (
    Path(r"C:\Users\chinn\OneDrive\Desktop\AI-Hospital-Readmission")
    / "app"
    / "pickles"
    / "model2"
)

encoder_path = (
    deployment_dir /
    "classification_onehot_encoder.pkl"
)

model_path = (
    deployment_dir /
    "final_xgboost_classifier.pkl"
)


# ============================================================
# 6. Create backups BEFORE replacing existing files
# ============================================================

encoder_backup_path = (
    deployment_dir /
    "classification_onehot_encoder_backup.pkl"
)

model_backup_path = (
    deployment_dir /
    "final_xgboost_classifier_backup.pkl"
)

if encoder_path.exists():
    shutil.copy2(
        encoder_path,
        encoder_backup_path
    )

if model_path.exists():
    shutil.copy2(
        model_path,
        model_backup_path
    )


# ============================================================
# 7. Save corrected standalone encoder
# ============================================================

with open(encoder_path, "wb") as file:
    pickle.dump(
        classification_encoder_final,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# ============================================================
# 8. Load existing final classification package
# ============================================================

with open(model_path, "rb") as file:
    classification_package = pickle.load(file)


# ============================================================
# 9. Replace ONLY the encoder inside the package
# ============================================================

classification_package["encoder"] = (
    classification_encoder_final
)


# ============================================================
# 10. Save corrected classification package
# ============================================================

with open(model_path, "wb") as file:
    pickle.dump(
        classification_package,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# ============================================================
# 11. Reload and verify
# ============================================================

with open(encoder_path, "rb") as file:
    verified_encoder = pickle.load(file)

with open(model_path, "rb") as file:
    verified_package = pickle.load(file)


assert hasattr(
    verified_encoder,
    "categories_"
)

assert verified_encoder.n_features_in_ == 17

assert len(
    verified_encoder.get_feature_names_out()
) == 201

assert hasattr(
    verified_package["encoder"],
    "categories_"
)

assert (
    verified_package["encoder"].n_features_in_
    == 17
)

assert len(
    verified_package["encoder"].get_feature_names_out()
) == 201


# ============================================================
# 12. Display final repair status
# ============================================================

repair_status = pd.DataFrame({
    "check": [
        "Encoder fitted",
        "Encoder input features",
        "Encoder output features",
        "Standalone encoder saved",
        "Package encoder saved",
        "Encoder backup created",
        "Model package backup created"
    ],
    "result": [
        True,
        verified_encoder.n_features_in_,
        len(verified_encoder.get_feature_names_out()),
        encoder_path.is_file(),
        model_path.is_file(),
        encoder_backup_path.is_file(),
        model_backup_path.is_file()
    ]
})

display(repair_status)

,check,result
0,Encoder fitted,True
1,Encoder input features,17
2,Encoder output features,201
3,Standalone encoder saved,True
4,Package encoder saved,True
5,Encoder backup created,True
6,Model package backup created,True
